# Core: Random Forest on laser-captured dopamine neurons in Parkinson's disease

**63 people** from four laser-capture studies (GSE182622, GSE20141, GSE24378, GSE169755),
one profile per person, each study z-scored on its own without looking at diagnosis.

**The model.** Each person's 5,622 genes are ranked within that person, the ranks are
compressed into 30 principal components (fitted on training people only), and a Random
Forest (1,000 trees, half the components tried at each split) classifies. It was the
best of 97 Random Forest variants in `pd-lcm-rf-confirm`; here it is re-scored on folds
that sweep never used.

**What this notebook saves** for the secondary notebooks: the data (`core_data.npz`),
the folds (`core_folds.json`), the fitted model (`core_model.joblib`), its performance,
its out-of-fold scores, and its SHAP importance for every component and every gene.

In [ ]:
import subprocess, sys, os, time, json, glob, warnings
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
for _a, _t in [("float", float), ("int", int), ("bool", bool), ("object", object), ("str", str)]:
    if not hasattr(np, _a):
        setattr(np, _a, _t)
ON_KAGGLE = Path("/kaggle/input").exists()
SMOKE = not ON_KAGGLE
OUT = Path("/kaggle/working") if ON_KAGGLE else Path(os.environ.get("SMOKE_OUT", "smoke_out"))
OUT.mkdir(exist_ok=True, parents=True)
LOCAL_IN = os.environ.get("SMOKE_IN", "data").split(":")
N_CPU = os.cpu_count()
SEED = 42
t0 = time.time()
def log(m): print(f"[{time.time() - t0:6.0f}s] {m}", flush=True)

def find_input(pattern):
    roots = ("/kaggle/input",) if ON_KAGGLE else tuple(LOCAL_IN)
    for root in roots:
        hits = sorted(glob.glob(f"{root}/**/{pattern}", recursive=True), key=len)
        if hits:
            return hits[0]
    raise FileNotFoundError(f"{pattern} not found under {roots}")
print("Kaggle" if ON_KAGGLE else "LOCAL SMOKE RUN", "| CPUs:", N_CPU)
import joblib, shap
from scipy.stats import rankdata
from joblib import Parallel, delayed
from sklearn.model_selection import StratifiedKFold
N_REP   = 1 if SMOKE else 5       # repeats of stratified 5-fold
N_TREES = 40 if SMOKE else 1000

In [ ]:
REF = json.loads(r'''{"person": ["C-04-52", "C-06-57", "C-07-28", "C-09-50", "C-10-39", "C-12-44", "C-14-42", "C-15-46", "C-15-60", "C-15-78", "PD-03-43", "PD-03-45", "PD-06-44", "PD-10-27", "PD-10-83", "PD-11-110", "PD-12-22", "PD-12-33", "PD-12-55", "PD-13-29", "PD-95-19", "PD-96-36", "C-1074p-SNc", "C-1271p-SNc", "C-2829-SNc", "C-3132p-SNc", "C-3397-SNc", "C-3543-SNc", "C-3603-SNc", "C-5220-SNc", "PD-1364-SNc", "PD-1401-SNc", "PD-1647-SNc", "PD-2515p-SNc", "PD-2525-SNc", "PD-3769p-SNc", "PD-3790p-SNc", "PD-3803p-SNc", "PD-5138-SNc", "PD-5476-SNc", "459A", "459B", "459C", "459D", "459E", "459F", "515B", "515C", "515D", "515E", "515F", "515G", "515H", "515I", "515J", "515K", "515L", "Brain2", "Brain3", "Brain4", "Brain7", "Brain11", "Brain12"], "ds": ["GSE182622", "GSE182622", "GSE182622", "GSE182622", "GSE182622", "GSE182622", "GSE182622", "GSE182622", "GSE182622", "GSE182622", "GSE182622", "GSE182622", "GSE182622", "GSE182622", "GSE182622", "GSE182622", "GSE182622", "GSE182622", "GSE182622", "GSE182622", "GSE182622", "GSE182622", "GSE20141", "GSE20141", "GSE20141", "GSE20141", "GSE20141", "GSE20141", "GSE20141", "GSE20141", "GSE20141", "GSE20141", "GSE20141", "GSE20141", "GSE20141", "GSE20141", "GSE20141", "GSE20141", "GSE20141", "GSE20141", "GSE24378", "GSE24378", "GSE24378", "GSE24378", "GSE24378", "GSE24378", "GSE24378", "GSE24378", "GSE24378", "GSE24378", "GSE24378", "GSE24378", "GSE24378", "GSE24378", "GSE24378", "GSE24378", "GSE24378", "GSE169755", "GSE169755", "GSE169755", "GSE169755", "GSE169755", "GSE169755"], "y": [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 1, 1, 1, 0, 1, 1, 1, 1, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 1], "genes": ["ENSG00000000003", "ENSG00000000419", "ENSG00000001036", "ENSG00000001084", "ENSG00000001460", "ENSG00000001461", "ENSG00000001497", "ENSG00000001561", "ENSG00000001629", "ENSG00000002549", "ENSG00000002746", "ENSG00000002834", "ENSG00000002919", "ENSG00000003096", "ENSG00000003147", "ENSG00000003393", "ENSG00000003402", "ENSG00000003509", "ENSG00000003756", "ENSG00000004142", "ENSG00000004455", "ENSG00000004534", "ENSG00000004660", "ENSG00000004766", "ENSG00000004779", "ENSG00000004799", "ENSG00000004864", "ENSG00000004897", "ENSG00000004961", "ENSG00000005007", "ENSG00000005020", "ENSG00000005022", "ENSG00000005100", "ENSG00000005108", "ENSG00000005175", "ENSG00000005194", "ENSG00000005249", "ENSG00000005302", "ENSG00000005339", "ENSG00000005448", "ENSG00000005483", "ENSG00000005486", "ENSG00000005700", "ENSG00000005801", "ENSG00000005810", "ENSG00000005812", "ENSG00000005893", "ENSG00000005981", "ENSG00000006007", "ENSG00000006116", "ENSG00000006118", "ENSG00000006125", "ENSG00000006128", "ENSG00000006210", "ENSG00000006282", "ENSG00000006283", "ENSG00000006451", "ENSG00000006530", "ENSG00000006625", "ENSG00000006704", "ENSG00000006712", "ENSG00000006715", "ENSG00000006740", "ENSG00000006744", "ENSG00000006831", "ENSG00000007047", "ENSG00000007168", "ENSG00000007392", "ENSG00000007402", "ENSG00000007944", "ENSG00000008018", "ENSG00000008056", "ENSG00000008083", "ENSG00000008256", "ENSG00000008277", "ENSG00000008282", "ENSG00000008283", "ENSG00000008294", "ENSG00000008300", "ENSG00000008324", "ENSG00000008394", "ENSG00000008513", "ENSG00000008735", "ENSG00000008869", "ENSG00000008952", "ENSG00000009307", "ENSG00000009335", "ENSG00000009413", "ENSG00000009694", "ENSG00000009844", "ENSG00000009954", "ENSG00000010017", "ENSG00000010244", "ENSG00000010270", "ENSG00000010295", "ENSG00000010322", "ENSG00000010810", "ENSG00000010818", "ENSG00000011009", "ENSG00000011021", "ENSG00000011114", "ENSG00000011275", "ENSG00000011295", "ENSG00000011376", "ENSG00000011405", "ENSG00000011426", "ENSG00000011451", "ENSG00000011485", "ENSG00000011523", "ENSG00000011566", "ENSG00000012061", "ENSG00000012174", "ENSG00000012232", "ENSG00000012660", "ENSG00000012822", "ENSG00000012963", "ENSG00000012983", "ENSG00000013016", "ENSG00000013275", "ENSG00000013288", "ENSG00000013293", "ENSG00000013374", "ENSG00000013375", "ENSG00000013392", "ENSG00000013441", "ENSG00000013523", "ENSG00000013561", "ENSG00000013588", "ENSG00000014123", "ENSG00000014216", "ENSG00000014641", "ENSG00000014824", "ENSG00000014919", "ENSG00000015153", "ENSG00000015171", "ENSG00000015532", "ENSG00000015592", "ENSG00000015676", "ENSG00000016864", "ENSG00000017260", "ENSG00000017427", "ENSG00000017797", "ENSG00000018189", "ENSG00000018236", "ENSG00000018510", "ENSG00000018625", "ENSG00000019505", "ENSG00000019582", "ENSG00000019995", "ENSG00000020129", "ENSG00000020426", "ENSG00000020577", "ENSG00000021300", "ENSG00000021355", "ENSG00000021574", "ENSG00000021645", "ENSG00000021762", "ENSG00000021776", "ENSG00000022267", "ENSG00000022355", "ENSG00000022567", "ENSG00000022840", "ENSG00000023041", "ENSG00000023171", "ENSG00000023191", "ENSG00000023228", "ENSG00000023287", "ENSG00000023318", "ENSG00000023516", "ENSG00000023572", "ENSG00000023734", "ENSG00000023909", "ENSG00000024048", "ENSG00000024862", "ENSG00000025039", "ENSG00000025156", "ENSG00000025293", "ENSG00000025770", "ENSG00000025772", "ENSG00000025796", "ENSG00000025800", "ENSG00000026652", "ENSG00000027697", "ENSG00000027847", "ENSG00000028203", "ENSG00000028310", "ENSG00000028528", "ENSG00000028839", "ENSG00000029363", "ENSG00000029364", "ENSG00000029534", "ENSG00000029725", "ENSG00000030582", "ENSG00000031003", "ENSG00000031823", "ENSG00000032219", "ENSG00000032444", "ENSG00000033122", "ENSG00000033170", "ENSG00000033178", "ENSG00000033327", "ENSG00000033627", "ENSG00000033867", "ENSG00000034053", "ENSG00000034152", "ENSG00000034510", "ENSG00000034677", "ENSG00000034713", "ENSG00000035115", "ENSG00000035403", "ENSG00000035681", "ENSG00000035687", "ENSG00000035862", "ENSG00000035928", "ENSG00000036054", "ENSG00000036257", "ENSG00000036530", "ENSG00000036549", "ENSG00000036565", "ENSG00000037042", "ENSG00000037474", "ENSG00000037637", "ENSG00000038219", "ENSG00000038274", "ENSG00000038382", "ENSG00000039319", "ENSG00000040199", "ENSG00000040341", "ENSG00000040731", "ENSG00000040933", "ENSG00000041353", "ENSG00000041357", "ENSG00000041802", "ENSG00000042317", "ENSG00000042753", "ENSG00000043093", "ENSG00000043143", "ENSG00000044115", "ENSG00000044574", "ENSG00000046653", "ENSG00000047056", "ENSG00000047188", "ENSG00000047249", "ENSG00000047315", "ENSG00000047410", "ENSG00000047579", "ENSG00000047597", "ENSG00000047648", "ENSG00000047849", "ENSG00000047932", "ENSG00000048052", "ENSG00000048392", "ENSG00000048471", "ENSG00000048540", "ENSG00000048544", "ENSG00000048649", "ENSG00000048707", "ENSG00000048740", "ENSG00000048828", "ENSG00000048991", "ENSG00000049245", "ENSG00000049618", "ENSG00000049656", "ENSG00000049759", "ENSG00000049769", "ENSG00000049860", "ENSG00000050130", "ENSG00000050165", "ENSG00000050393", "ENSG00000050426", "ENSG00000050438", "ENSG00000050748", "ENSG00000050820", "ENSG00000051108", "ENSG00000051620", "ENSG00000051825", "ENSG00000052126", "ENSG00000052723", "ENSG00000052749", "ENSG00000052795", "ENSG00000052802", "ENSG00000052841", "ENSG00000053254", "ENSG00000053372", "ENSG00000053501", "ENSG00000053524", "ENSG00000053747", "ENSG00000053770", "ENSG00000053900", "ENSG00000054116", "ENSG00000054118", "ENSG00000054267", "ENSG00000054282", "ENSG00000054356", "ENSG00000054523", "ENSG00000054611", "ENSG00000054690", "ENSG00000054793", "ENSG00000054965", "ENSG00000054983", "ENSG00000055044", "ENSG00000055118", "ENSG00000055130", "ENSG00000055147", "ENSG00000055163", "ENSG00000055208", "ENSG00000055211", "ENSG00000055332", "ENSG00000055483", "ENSG00000055609", "ENSG00000055917", "ENSG00000055950", "ENSG00000056097", "ENSG00000056586", "ENSG00000056998", "ENSG00000057019", "ENSG00000057252", "ENSG00000057608", "ENSG00000057663", "ENSG00000057704", "ENSG00000057757", "ENSG00000057935", "ENSG00000058091", "ENSG00000058272", "ENSG00000058335", "ENSG00000058404", "ENSG00000058600", "ENSG00000058729", "ENSG00000058799", "ENSG00000059122", "ENSG00000059145", "ENSG00000059573", "ENSG00000059691", "ENSG00000059728", "ENSG00000059804", "ENSG00000059915", "ENSG00000060237", "ENSG00000060339", "ENSG00000060491", "ENSG00000060656", "ENSG00000060709", "ENSG00000060718", "ENSG00000060762", "ENSG00000060982", "ENSG00000061676", "ENSG00000061794", "ENSG00000061936", "ENSG00000061938", "ENSG00000061987", "ENSG00000062194", "ENSG00000062598", "ENSG00000062650", "ENSG00000062716", "ENSG00000062725", "ENSG00000063046", "ENSG00000063176", "ENSG00000063177", "ENSG00000063244", "ENSG00000063245", "ENSG00000063322", "ENSG00000063587", "ENSG00000063601", "ENSG00000063660", "ENSG00000063854", "ENSG00000063978", "ENSG00000064042", "ENSG00000064115", "ENSG00000064225", "ENSG00000064313", "ENSG00000064393", "ENSG00000064601", "ENSG00000064607", "ENSG00000064651", "ENSG00000064652", "ENSG00000064687", "ENSG00000064726", "ENSG00000064787", "ENSG00000064933", "ENSG00000064999", "ENSG00000065000", "ENSG00000065135", "ENSG00000065150", "ENSG00000065154", "ENSG00000065183", "ENSG00000065457", "ENSG00000065491", "ENSG00000065518", "ENSG00000065526", "ENSG00000065534", "ENSG00000065548", "ENSG00000065559", "ENSG00000065609", "ENSG00000065613", "ENSG00000065665", "ENSG00000065802", "ENSG00000065809", "ENSG00000065833", "ENSG00000065883", "ENSG00000065911", "ENSG00000065989", "ENSG00000066027", "ENSG00000066032", "ENSG00000066044", "ENSG00000066084", "ENSG00000066117", "ENSG00000066136", "ENSG00000066382", "ENSG00000066422", "ENSG00000066427", "ENSG00000066455", "ENSG00000066468", "ENSG00000066557", "ENSG00000066583", "ENSG00000066651", "ENSG00000066654", "ENSG00000066697", "ENSG00000066739", "ENSG00000066777", "ENSG00000066926", "ENSG00000066933", "ENSG00000067048", "ENSG00000067057", "ENSG00000067064", "ENSG00000067082", "ENSG00000067141", "ENSG00000067177", "ENSG00000067191", "ENSG00000067208", "ENSG00000067221", "ENSG00000067225", "ENSG00000067248", "ENSG00000067334", "ENSG00000067369", "ENSG00000067445", "ENSG00000067533", "ENSG00000067560", "ENSG00000067606", "ENSG00000067704", "ENSG00000067715", "ENSG00000067798", "ENSG00000067829", "ENSG00000067836", "ENSG00000067840", "ENSG00000067842", "ENSG00000067955", "ENSG00000068024", "ENSG00000068078", "ENSG00000068323", "ENSG00000068366", "ENSG00000068383", "ENSG00000068400", "ENSG00000068438", "ENSG00000068615", "ENSG00000068650", "ENSG00000068654", "ENSG00000068697", "ENSG00000068745", "ENSG00000068796", "ENSG00000068878", "ENSG00000068885", "ENSG00000068903", "ENSG00000068912", "ENSG00000068971", "ENSG00000069020", "ENSG00000069248", "ENSG00000069275", "ENSG00000069345", "ENSG00000069424", "ENSG00000069509", "ENSG00000069535", "ENSG00000069667", "ENSG00000069849", "ENSG00000069956", "ENSG00000069966", "ENSG00000070010", "ENSG00000070018", "ENSG00000070081", "ENSG00000070087", "ENSG00000070182", "ENSG00000070214", "ENSG00000070367", "ENSG00000070371", "ENSG00000070413", "ENSG00000070423", "ENSG00000070444", "ENSG00000070495", "ENSG00000070501", "ENSG00000070610", "ENSG00000070614", "ENSG00000070718", "ENSG00000070756", "ENSG00000070761", "ENSG00000070770", "ENSG00000070785", "ENSG00000070814", "ENSG00000070882", "ENSG00000070961", "ENSG00000071051", "ENSG00000071054", "ENSG00000071082", "ENSG00000071127", "ENSG00000071189", "ENSG00000071537", "ENSG00000071553", "ENSG00000071575", "ENSG00000071626", "ENSG00000071655", "ENSG00000071794", "ENSG00000071859", "ENSG00000071889", "ENSG00000071994", "ENSG00000072041", "ENSG00000072042", "ENSG00000072062", "ENSG00000072071", "ENSG00000072110", "ENSG00000072134", "ENSG00000072135", "ENSG00000072195", "ENSG00000072210", "ENSG00000072274", "ENSG00000072364", "ENSG00000072401", "ENSG00000072501", "ENSG00000072756", "ENSG00000072778", "ENSG00000072803", "ENSG00000072832", "ENSG00000072849", "ENSG00000072858", "ENSG00000072954", "ENSG00000073008", "ENSG00000073150", "ENSG00000073464", "ENSG00000073614", "ENSG00000073803", "ENSG00000073910", "ENSG00000073921", "ENSG00000073969", "ENSG00000074054", "ENSG00000074071", "ENSG00000074201", "ENSG00000074211", "ENSG00000074319", "ENSG00000074370", "ENSG00000074416", "ENSG00000074527", "ENSG00000074582", "ENSG00000074590", "ENSG00000074603", "ENSG00000074695", "ENSG00000074696", "ENSG00000074706", "ENSG00000074800", "ENSG00000074842", "ENSG00000074935", "ENSG00000074964", "ENSG00000075035", "ENSG00000075043", "ENSG00000075089", "ENSG00000075142", "ENSG00000075151", "ENSG00000075239", "ENSG00000075240", "ENSG00000075292", "ENSG00000075336", "ENSG00000075340", "ENSG00000075391", "ENSG00000075399", "ENSG00000075407", "ENSG00000075413", "ENSG00000075415", "ENSG00000075539", "ENSG00000075568", "ENSG00000075618", "ENSG00000075624", "ENSG00000075711", "ENSG00000075785", "ENSG00000075856", "ENSG00000075914", "ENSG00000075945", "ENSG00000075975", "ENSG00000076043", "ENSG00000076108", "ENSG00000076242", "ENSG00000076248", "ENSG00000076321", "ENSG00000076513", "ENSG00000076554", "ENSG00000076641", "ENSG00000076716", "ENSG00000076864", "ENSG00000076984", "ENSG00000077044", "ENSG00000077063", "ENSG00000077097", "ENSG00000077147", "ENSG00000077157", "ENSG00000077232", "ENSG00000077235", "ENSG00000077254", "ENSG00000077264", "ENSG00000077279", "ENSG00000077312", "ENSG00000077380", "ENSG00000077549", "ENSG00000077585", "ENSG00000077616", "ENSG00000077684", "ENSG00000077721", "ENSG00000077782", "ENSG00000078018", "ENSG00000078043", "ENSG00000078053", "ENSG00000078114", "ENSG00000078124", "ENSG00000078140", "ENSG00000078142", "ENSG00000078295", "ENSG00000078304", "ENSG00000078328", "ENSG00000078369", "ENSG00000078549", "ENSG00000078596", "ENSG00000078668", "ENSG00000078674", "ENSG00000078687", "ENSG00000078699", "ENSG00000078804", "ENSG00000078967", "ENSG00000079102", "ENSG00000079156", "ENSG00000079215", "ENSG00000079246", "ENSG00000079277", "ENSG00000079308", "ENSG00000079332", "ENSG00000079459", "ENSG00000079462", "ENSG00000079739", "ENSG00000079785", "ENSG00000079841", "ENSG00000079950", "ENSG00000079999", "ENSG00000080189", "ENSG00000080345", "ENSG00000080371", "ENSG00000080493", "ENSG00000080503", "ENSG00000080546", "ENSG00000080802", "ENSG00000080815", "ENSG00000080819", "ENSG00000080822", "ENSG00000080824", "ENSG00000080845", "ENSG00000081019", "ENSG00000081026", "ENSG00000081087", "ENSG00000081154", "ENSG00000081307", "ENSG00000081760", "ENSG00000081803", "ENSG00000082014", "ENSG00000082153", "ENSG00000082212", "ENSG00000082213", "ENSG00000082258", "ENSG00000082269", "ENSG00000082397", "ENSG00000082458", "ENSG00000082497", "ENSG00000082515", "ENSG00000082516", "ENSG00000082556", "ENSG00000082641", "ENSG00000082701", "ENSG00000082805", "ENSG00000082898", "ENSG00000082996", "ENSG00000083099", "ENSG00000083123", "ENSG00000083168", "ENSG00000083290", "ENSG00000083312", "ENSG00000083444", "ENSG00000083454", "ENSG00000083457", "ENSG00000083520", "ENSG00000083642", "ENSG00000083720", "ENSG00000083750", "ENSG00000083799", "ENSG00000083845", "ENSG00000083937", "ENSG00000084070", "ENSG00000084073", "ENSG00000084090", "ENSG00000084112", "ENSG00000084207", "ENSG00000084234", "ENSG00000084463", "ENSG00000084623", "ENSG00000084652", "ENSG00000084676", "ENSG00000084693", "ENSG00000084731", "ENSG00000084733", "ENSG00000084754", "ENSG00000084764", "ENSG00000085224", "ENSG00000085231", "ENSG00000085274", "ENSG00000085365", "ENSG00000085377", "ENSG00000085382", "ENSG00000085433", "ENSG00000085449", "ENSG00000085511", "ENSG00000085644", "ENSG00000085719", "ENSG00000085721", "ENSG00000085733", "ENSG00000085788", "ENSG00000085831", "ENSG00000085832", "ENSG00000085871", "ENSG00000085872", "ENSG00000085978", "ENSG00000085998", "ENSG00000086015", "ENSG00000086061", "ENSG00000086065", "ENSG00000086102", "ENSG00000086189", "ENSG00000086200", "ENSG00000086232", "ENSG00000086289", "ENSG00000086300", "ENSG00000086475", "ENSG00000086504", "ENSG00000086589", "ENSG00000086598", "ENSG00000086619", "ENSG00000086758", "ENSG00000087053", "ENSG00000087087", "ENSG00000087095", "ENSG00000087152", "ENSG00000087191", "ENSG00000087258", "ENSG00000087263", "ENSG00000087274", "ENSG00000087302", "ENSG00000087338", "ENSG00000087365", "ENSG00000087448", "ENSG00000087460", "ENSG00000087470", "ENSG00000087495", "ENSG00000087502", "ENSG00000088179", "ENSG00000088205", "ENSG00000088247", "ENSG00000088256", "ENSG00000088356", "ENSG00000088367", "ENSG00000088387", "ENSG00000088448", "ENSG00000088538", "ENSG00000088543", "ENSG00000088808", "ENSG00000088812", "ENSG00000088833", "ENSG00000088899", "ENSG00000088930", "ENSG00000088986", "ENSG00000089006", "ENSG00000089048", "ENSG00000089053", "ENSG00000089057", "ENSG00000089154", "ENSG00000089169", "ENSG00000089195", "ENSG00000089199", "ENSG00000089220", "ENSG00000089234", "ENSG00000089248", "ENSG00000089280", "ENSG00000089289", "ENSG00000089335", "ENSG00000089486", "ENSG00000089693", "ENSG00000089737", "ENSG00000089818", "ENSG00000089916", "ENSG00000090013", "ENSG00000090020", "ENSG00000090060", "ENSG00000090097", "ENSG00000090238", "ENSG00000090263", "ENSG00000090266", "ENSG00000090432", "ENSG00000090470", "ENSG00000090487", "ENSG00000090565", "ENSG00000090615", "ENSG00000090621", "ENSG00000090686", "ENSG00000090863", "ENSG00000090905", "ENSG00000090971", "ENSG00000090975", "ENSG00000090989", "ENSG00000091009", "ENSG00000091039", "ENSG00000091129", "ENSG00000091140", "ENSG00000091157", "ENSG00000091164", "ENSG00000091428", "ENSG00000091482", "ENSG00000091513", "ENSG00000091527", "ENSG00000091542", "ENSG00000091640", "ENSG00000091656", "ENSG00000091732", "ENSG00000091844", "ENSG00000091947", "ENSG00000091972", "ENSG00000092010", "ENSG00000092020", "ENSG00000092096", "ENSG00000092108", "ENSG00000092140", "ENSG00000092148", "ENSG00000092199", "ENSG00000092201", "ENSG00000092203", "ENSG00000092330", "ENSG00000092820", "ENSG00000092841", "ENSG00000092847", "ENSG00000092931", "ENSG00000092964", "ENSG00000092978", "ENSG00000093000", "ENSG00000093010", "ENSG00000093144", "ENSG00000093167", "ENSG00000093183", "ENSG00000094631", "ENSG00000094841", "ENSG00000094880", "ENSG00000094916", "ENSG00000094975", "ENSG00000095139", "ENSG00000095261", "ENSG00000095321", "ENSG00000095380", "ENSG00000095564", "ENSG00000095574", "ENSG00000095637", "ENSG00000095787", "ENSG00000095794", "ENSG00000096063", "ENSG00000096092", "ENSG00000096384", "ENSG00000096401", "ENSG00000096746", "ENSG00000097007", "ENSG00000097021", "ENSG00000097033", "ENSG00000099194", "ENSG00000099204", "ENSG00000099219", "ENSG00000099246", "ENSG00000099308", "ENSG00000099326", "ENSG00000099341", "ENSG00000099622", "ENSG00000099783", "ENSG00000099814", "ENSG00000099817", "ENSG00000099864", "ENSG00000099875", "ENSG00000099889", "ENSG00000099901", "ENSG00000099904", "ENSG00000099910", "ENSG00000099917", "ENSG00000099940", "ENSG00000099942", "ENSG00000099956", "ENSG00000099968", "ENSG00000099991", "ENSG00000099995", "ENSG00000100014", "ENSG00000100029", "ENSG00000100030", "ENSG00000100034", "ENSG00000100083", "ENSG00000100095", "ENSG00000100097", "ENSG00000100099", "ENSG00000100100", "ENSG00000100116", "ENSG00000100124", "ENSG00000100138", "ENSG00000100142", "ENSG00000100151", "ENSG00000100154", "ENSG00000100201", "ENSG00000100207", "ENSG00000100216", "ENSG00000100220", "ENSG00000100221", "ENSG00000100225", "ENSG00000100227", "ENSG00000100234", "ENSG00000100239", "ENSG00000100241", "ENSG00000100243", "ENSG00000100263", "ENSG00000100266", "ENSG00000100280", "ENSG00000100281", "ENSG00000100304", "ENSG00000100307", "ENSG00000100320", "ENSG00000100321", "ENSG00000100324", "ENSG00000100330", "ENSG00000100347", "ENSG00000100348", "ENSG00000100350", "ENSG00000100353", "ENSG00000100354", "ENSG00000100364", "ENSG00000100372", "ENSG00000100379", "ENSG00000100380", "ENSG00000100387", "ENSG00000100395", "ENSG00000100401", "ENSG00000100403", "ENSG00000100413", "ENSG00000100417", "ENSG00000100418", "ENSG00000100422", "ENSG00000100425", "ENSG00000100442", "ENSG00000100461", "ENSG00000100483", "ENSG00000100485", "ENSG00000100503", "ENSG00000100504", "ENSG00000100505", "ENSG00000100523", "ENSG00000100528", "ENSG00000100554", "ENSG00000100564", "ENSG00000100567", "ENSG00000100575", "ENSG00000100580", "ENSG00000100591", "ENSG00000100592", "ENSG00000100596", "ENSG00000100600", "ENSG00000100603", "ENSG00000100604", "ENSG00000100605", "ENSG00000100612", "ENSG00000100614", "ENSG00000100626", "ENSG00000100632", "ENSG00000100644", "ENSG00000100647", "ENSG00000100650", "ENSG00000100664", "ENSG00000100678", "ENSG00000100711", "ENSG00000100714", "ENSG00000100744", "ENSG00000100784", "ENSG00000100796", "ENSG00000100804", "ENSG00000100811", "ENSG00000100813", "ENSG00000100815", "ENSG00000100823", "ENSG00000100852", "ENSG00000100865", "ENSG00000100883", "ENSG00000100888", "ENSG00000100897", "ENSG00000100906", "ENSG00000100908", "ENSG00000100934", "ENSG00000100938", "ENSG00000100941", "ENSG00000100983", "ENSG00000100997", "ENSG00000101019", "ENSG00000101040", "ENSG00000101079", "ENSG00000101109", "ENSG00000101126", "ENSG00000101132", "ENSG00000101134", "ENSG00000101146", "ENSG00000101150", "ENSG00000101152", "ENSG00000101161", "ENSG00000101182", "ENSG00000101187", "ENSG00000101189", "ENSG00000101190", "ENSG00000101191", "ENSG00000101193", "ENSG00000101199", "ENSG00000101210", "ENSG00000101224", "ENSG00000101236", "ENSG00000101246", "ENSG00000101247", "ENSG00000101265", "ENSG00000101266", "ENSG00000101290", "ENSG00000101298", "ENSG00000101310", "ENSG00000101333", "ENSG00000101343", "ENSG00000101347", "ENSG00000101350", "ENSG00000101363", "ENSG00000101365", "ENSG00000101367", "ENSG00000101391", "ENSG00000101407", "ENSG00000101413", "ENSG00000101421", "ENSG00000101439", "ENSG00000101460", "ENSG00000101464", "ENSG00000101474", "ENSG00000101489", "ENSG00000101544", "ENSG00000101557", "ENSG00000101558", "ENSG00000101577", "ENSG00000101596", "ENSG00000101608", "ENSG00000101654", "ENSG00000101745", "ENSG00000101746", "ENSG00000101751", "ENSG00000101752", "ENSG00000101782", "ENSG00000101843", "ENSG00000101846", "ENSG00000101856", "ENSG00000101882", "ENSG00000101928", "ENSG00000101940", "ENSG00000101966", "ENSG00000101972", "ENSG00000101974", "ENSG00000101977", "ENSG00000102003", "ENSG00000102024", "ENSG00000102030", "ENSG00000102038", "ENSG00000102054", "ENSG00000102078", "ENSG00000102081", "ENSG00000102100", "ENSG00000102103", "ENSG00000102109", "ENSG00000102119", "ENSG00000102144", "ENSG00000102178", "ENSG00000102181", "ENSG00000102189", "ENSG00000102225", "ENSG00000102226", "ENSG00000102241", "ENSG00000102309", "ENSG00000102316", "ENSG00000102393", "ENSG00000102401", "ENSG00000102409", "ENSG00000102452", "ENSG00000102466", "ENSG00000102468", "ENSG00000102471", "ENSG00000102524", "ENSG00000102531", "ENSG00000102547", "ENSG00000102572", "ENSG00000102606", "ENSG00000102678", "ENSG00000102753", "ENSG00000102760", "ENSG00000102763", "ENSG00000102781", "ENSG00000102804", "ENSG00000102858", "ENSG00000102882", "ENSG00000102893", "ENSG00000102897", "ENSG00000102898", "ENSG00000102900", "ENSG00000102908", "ENSG00000102910", "ENSG00000102921", "ENSG00000102924", "ENSG00000102931", "ENSG00000102935", "ENSG00000102974", "ENSG00000102978", "ENSG00000102981", "ENSG00000103005", "ENSG00000103018", "ENSG00000103034", "ENSG00000103035", "ENSG00000103042", "ENSG00000103051", "ENSG00000103064", "ENSG00000103091", "ENSG00000103111", "ENSG00000103121", "ENSG00000103145", "ENSG00000103148", "ENSG00000103150", "ENSG00000103160", "ENSG00000103222", "ENSG00000103253", "ENSG00000103257", "ENSG00000103264", "ENSG00000103275", "ENSG00000103342", "ENSG00000103351", "ENSG00000103353", "ENSG00000103365", "ENSG00000103381", "ENSG00000103404", "ENSG00000103415", "ENSG00000103423", "ENSG00000103429", "ENSG00000103460", "ENSG00000103485", "ENSG00000103489", "ENSG00000103496", "ENSG00000103502", "ENSG00000103528", "ENSG00000103540", "ENSG00000103591", "ENSG00000103647", "ENSG00000103653", "ENSG00000103657", "ENSG00000103723", "ENSG00000103769", "ENSG00000103876", "ENSG00000103978", "ENSG00000103994", "ENSG00000104067", "ENSG00000104093", "ENSG00000104112", "ENSG00000104131", "ENSG00000104133", "ENSG00000104142", "ENSG00000104164", "ENSG00000104177", "ENSG00000104218", "ENSG00000104219", "ENSG00000104231", "ENSG00000104267", "ENSG00000104290", "ENSG00000104320", "ENSG00000104327", "ENSG00000104332", "ENSG00000104341", "ENSG00000104343", "ENSG00000104361", "ENSG00000104381", "ENSG00000104388", "ENSG00000104408", "ENSG00000104412", "ENSG00000104427", "ENSG00000104435", "ENSG00000104442", "ENSG00000104447", "ENSG00000104490", "ENSG00000104517", "ENSG00000104613", "ENSG00000104635", "ENSG00000104643", "ENSG00000104660", "ENSG00000104671", "ENSG00000104679", "ENSG00000104687", "ENSG00000104691", "ENSG00000104695", "ENSG00000104722", "ENSG00000104723", "ENSG00000104738", "ENSG00000104756", "ENSG00000104763", "ENSG00000104765", "ENSG00000104805", "ENSG00000104823", "ENSG00000104833", "ENSG00000104852", "ENSG00000104853", "ENSG00000104863", "ENSG00000104880", "ENSG00000104884", "ENSG00000104904", "ENSG00000104915", "ENSG00000104967", "ENSG00000104969", "ENSG00000104979", "ENSG00000104980", "ENSG00000105058", "ENSG00000105088", "ENSG00000105171", "ENSG00000105185", "ENSG00000105186", "ENSG00000105193", "ENSG00000105202", "ENSG00000105204", "ENSG00000105220", "ENSG00000105245", "ENSG00000105254", "ENSG00000105255", "ENSG00000105258", "ENSG00000105270", "ENSG00000105278", "ENSG00000105290", "ENSG00000105323", "ENSG00000105325", "ENSG00000105339", "ENSG00000105357", "ENSG00000105364", "ENSG00000105372", "ENSG00000105379", "ENSG00000105426", "ENSG00000105429", "ENSG00000105438", "ENSG00000105447", "ENSG00000105499", "ENSG00000105516", "ENSG00000105518", "ENSG00000105568", "ENSG00000105576", "ENSG00000105618", "ENSG00000105649", "ENSG00000105671", "ENSG00000105698", "ENSG00000105700", "ENSG00000105701", "ENSG00000105705", "ENSG00000105723", "ENSG00000105737", "ENSG00000105771", "ENSG00000105778", "ENSG00000105784", "ENSG00000105793", "ENSG00000105819", "ENSG00000105835", "ENSG00000105854", "ENSG00000105939", "ENSG00000105983", "ENSG00000105993", "ENSG00000106028", "ENSG00000106049", "ENSG00000106052", "ENSG00000106070", "ENSG00000106077", "ENSG00000106078", "ENSG00000106086", "ENSG00000106123", "ENSG00000106144", "ENSG00000106244", "ENSG00000106261", "ENSG00000106263", "ENSG00000106266", "ENSG00000106278", "ENSG00000106299", "ENSG00000106344", "ENSG00000106348", "ENSG00000106351", "ENSG00000106355", "ENSG00000106392", "ENSG00000106399", "ENSG00000106400", "ENSG00000106443", "ENSG00000106460", "ENSG00000106477", "ENSG00000106484", "ENSG00000106524", "ENSG00000106537", "ENSG00000106591", "ENSG00000106603", "ENSG00000106605", "ENSG00000106608", "ENSG00000106609", "ENSG00000106615", "ENSG00000106617", "ENSG00000106628", "ENSG00000106635", "ENSG00000106665", "ENSG00000106682", "ENSG00000106683", "ENSG00000106688", "ENSG00000106692", "ENSG00000106701", "ENSG00000106723", "ENSG00000106733", "ENSG00000106771", "ENSG00000106772", "ENSG00000106780", "ENSG00000106789", "ENSG00000106799", "ENSG00000106803", "ENSG00000106829", "ENSG00000106976", "ENSG00000106993", "ENSG00000107021", "ENSG00000107077", "ENSG00000107104", "ENSG00000107105", "ENSG00000107130", "ENSG00000107164", "ENSG00000107185", "ENSG00000107186", "ENSG00000107223", "ENSG00000107242", "ENSG00000107262", "ENSG00000107263", "ENSG00000107290", "ENSG00000107295", "ENSG00000107331", "ENSG00000107341", "ENSG00000107362", "ENSG00000107372", "ENSG00000107404", "ENSG00000107518", "ENSG00000107537", "ENSG00000107551", "ENSG00000107560", "ENSG00000107581", "ENSG00000107625", "ENSG00000107643", "ENSG00000107651", "ENSG00000107669", "ENSG00000107742", "ENSG00000107745", "ENSG00000107758", "ENSG00000107771", "ENSG00000107798", "ENSG00000107819", "ENSG00000107829", "ENSG00000107854", "ENSG00000107862", "ENSG00000107864", "ENSG00000107874", "ENSG00000107897", "ENSG00000107902", "ENSG00000107929", "ENSG00000107937", "ENSG00000107949", "ENSG00000107951", "ENSG00000107954", "ENSG00000107957", "ENSG00000107959", "ENSG00000108001", "ENSG00000108039", "ENSG00000108055", "ENSG00000108061", "ENSG00000108091", "ENSG00000108094", "ENSG00000108100", "ENSG00000108107", "ENSG00000108175", "ENSG00000108176", "ENSG00000108179", "ENSG00000108231", "ENSG00000108256", "ENSG00000108262", "ENSG00000108298", "ENSG00000108306", "ENSG00000108309", "ENSG00000108312", "ENSG00000108344", "ENSG00000108349", "ENSG00000108352", "ENSG00000108379", "ENSG00000108384", "ENSG00000108389", "ENSG00000108395", "ENSG00000108406", "ENSG00000108424", "ENSG00000108433", "ENSG00000108443", "ENSG00000108468", "ENSG00000108509", "ENSG00000108510", "ENSG00000108528", "ENSG00000108551", "ENSG00000108582", "ENSG00000108587", "ENSG00000108588", "ENSG00000108599", "ENSG00000108654", "ENSG00000108669", "ENSG00000108671", "ENSG00000108679", "ENSG00000108684", "ENSG00000108773", "ENSG00000108788", "ENSG00000108797", "ENSG00000108799", "ENSG00000108819", "ENSG00000108826", "ENSG00000108828", "ENSG00000108829", "ENSG00000108830", "ENSG00000108848", "ENSG00000108852", "ENSG00000108861", "ENSG00000108883", "ENSG00000108946", "ENSG00000108953", "ENSG00000108960", "ENSG00000109046", "ENSG00000109065", "ENSG00000109099", "ENSG00000109103", "ENSG00000109107", "ENSG00000109111", "ENSG00000109118", "ENSG00000109133", "ENSG00000109158", "ENSG00000109171", "ENSG00000109180", "ENSG00000109184", "ENSG00000109189", "ENSG00000109270", "ENSG00000109332", "ENSG00000109339", "ENSG00000109390", "ENSG00000109436", "ENSG00000109445", "ENSG00000109452", "ENSG00000109458", "ENSG00000109466", "ENSG00000109472", "ENSG00000109475", "ENSG00000109501", "ENSG00000109519", "ENSG00000109572", "ENSG00000109606", "ENSG00000109654", "ENSG00000109670", "ENSG00000109680", "ENSG00000109689", "ENSG00000109738", "ENSG00000109756", "ENSG00000109762", "ENSG00000109790", "ENSG00000109832", "ENSG00000109846", "ENSG00000109854", "ENSG00000109881", "ENSG00000109917", "ENSG00000109919", "ENSG00000109920", "ENSG00000109929", "ENSG00000109956", "ENSG00000110002", "ENSG00000110013", "ENSG00000110042", "ENSG00000110046", "ENSG00000110047", "ENSG00000110048", "ENSG00000110074", "ENSG00000110075", "ENSG00000110076", "ENSG00000110107", "ENSG00000110172", "ENSG00000110237", "ENSG00000110274", "ENSG00000110315", "ENSG00000110318", "ENSG00000110321", "ENSG00000110330", "ENSG00000110344", "ENSG00000110367", "ENSG00000110395", "ENSG00000110422", "ENSG00000110427", "ENSG00000110429", "ENSG00000110435", "ENSG00000110436", "ENSG00000110442", "ENSG00000110497", "ENSG00000110514", "ENSG00000110651", "ENSG00000110675", "ENSG00000110693", "ENSG00000110696", "ENSG00000110697", "ENSG00000110713", "ENSG00000110717", "ENSG00000110768", "ENSG00000110786", "ENSG00000110841", "ENSG00000110851", "ENSG00000110880", "ENSG00000110881", "ENSG00000110888", "ENSG00000110906", "ENSG00000110911", "ENSG00000110917", "ENSG00000110921", "ENSG00000110925", "ENSG00000110931", "ENSG00000110958", "ENSG00000110987", "ENSG00000111011", "ENSG00000111052", "ENSG00000111077", "ENSG00000111110", "ENSG00000111142", "ENSG00000111144", "ENSG00000111218", "ENSG00000111237", "ENSG00000111249", "ENSG00000111261", "ENSG00000111269", "ENSG00000111275", "ENSG00000111276", "ENSG00000111300", "ENSG00000111358", "ENSG00000111361", "ENSG00000111371", "ENSG00000111481", "ENSG00000111530", "ENSG00000111596", "ENSG00000111605", "ENSG00000111639", "ENSG00000111640", "ENSG00000111642", "ENSG00000111652", "ENSG00000111653", "ENSG00000111666", "ENSG00000111669", "ENSG00000111670", "ENSG00000111674", "ENSG00000111676", "ENSG00000111696", "ENSG00000111707", "ENSG00000111711", "ENSG00000111726", "ENSG00000111728", "ENSG00000111731", "ENSG00000111785", "ENSG00000111786", "ENSG00000111790", "ENSG00000111799", "ENSG00000111802", "ENSG00000111832", "ENSG00000111843", "ENSG00000111845", "ENSG00000111860", "ENSG00000111875", "ENSG00000111879", "ENSG00000111880", "ENSG00000111897", "ENSG00000111906", "ENSG00000111907", "ENSG00000111911", "ENSG00000111912", "ENSG00000111961", "ENSG00000111962", "ENSG00000112062", "ENSG00000112078", "ENSG00000112079", "ENSG00000112081", "ENSG00000112110", "ENSG00000112130", "ENSG00000112146", "ENSG00000112149", "ENSG00000112159", "ENSG00000112186", "ENSG00000112200", "ENSG00000112234", "ENSG00000112237", "ENSG00000112242", "ENSG00000112249", "ENSG00000112282", "ENSG00000112290", "ENSG00000112294", "ENSG00000112304", "ENSG00000112305", "ENSG00000112308", "ENSG00000112312", "ENSG00000112320", "ENSG00000112335", "ENSG00000112339", "ENSG00000112367", "ENSG00000112378", "ENSG00000112379", "ENSG00000112406", "ENSG00000112419", "ENSG00000112425", "ENSG00000112531", "ENSG00000112576", "ENSG00000112592", "ENSG00000112640", "ENSG00000112651", "ENSG00000112658", "ENSG00000112659", "ENSG00000112667", "ENSG00000112679", "ENSG00000112685", "ENSG00000112695", "ENSG00000112697", "ENSG00000112699", "ENSG00000112701", "ENSG00000112715", "ENSG00000112763", "ENSG00000112796", "ENSG00000112855", "ENSG00000112874", "ENSG00000112893", "ENSG00000112972", "ENSG00000112981", "ENSG00000112983", "ENSG00000112992", "ENSG00000112996", "ENSG00000113013", "ENSG00000113048", "ENSG00000113068", "ENSG00000113140", "ENSG00000113141", "ENSG00000113161", "ENSG00000113194", "ENSG00000113240", "ENSG00000113269", "ENSG00000113282", "ENSG00000113312", "ENSG00000113327", "ENSG00000113328", "ENSG00000113360", "ENSG00000113361", "ENSG00000113384", "ENSG00000113387", "ENSG00000113441", "ENSG00000113448", "ENSG00000113456", "ENSG00000113504", "ENSG00000113552", "ENSG00000113569", "ENSG00000113575", "ENSG00000113578", "ENSG00000113580", "ENSG00000113583", "ENSG00000113594", "ENSG00000113595", "ENSG00000113597", "ENSG00000113615", "ENSG00000113621", "ENSG00000113657", "ENSG00000113658", "ENSG00000113719", "ENSG00000113732", "ENSG00000113734", "ENSG00000113742", "ENSG00000113758", "ENSG00000113845", "ENSG00000113851", "ENSG00000113916", "ENSG00000113966", "ENSG00000114019", "ENSG00000114021", "ENSG00000114023", "ENSG00000114030", "ENSG00000114054", "ENSG00000114062", "ENSG00000114098", "ENSG00000114107", "ENSG00000114120", "ENSG00000114125", "ENSG00000114126", "ENSG00000114127", "ENSG00000114166", "ENSG00000114209", "ENSG00000114251", "ENSG00000114279", "ENSG00000114302", "ENSG00000114316", "ENSG00000114353", "ENSG00000114354", "ENSG00000114388", "ENSG00000114416", "ENSG00000114439", "ENSG00000114446", "ENSG00000114480", "ENSG00000114503", "ENSG00000114520", "ENSG00000114541", "ENSG00000114544", "ENSG00000114554", "ENSG00000114573", "ENSG00000114646", "ENSG00000114648", "ENSG00000114650", "ENSG00000114686", "ENSG00000114742", "ENSG00000114757", "ENSG00000114770", "ENSG00000114796", "ENSG00000114805", "ENSG00000114850", "ENSG00000114853", "ENSG00000114857", "ENSG00000114867", "ENSG00000114902", "ENSG00000114904", "ENSG00000114923", "ENSG00000114948", "ENSG00000114956", "ENSG00000114978", "ENSG00000114982", "ENSG00000114988", "ENSG00000114993", "ENSG00000114999", "ENSG00000115020", "ENSG00000115073", "ENSG00000115084", "ENSG00000115091", "ENSG00000115109", "ENSG00000115128", "ENSG00000115137", "ENSG00000115145", "ENSG00000115159", "ENSG00000115170", "ENSG00000115204", "ENSG00000115211", "ENSG00000115233", "ENSG00000115239", "ENSG00000115252", "ENSG00000115266", "ENSG00000115271", "ENSG00000115295", "ENSG00000115306", "ENSG00000115307", "ENSG00000115310", "ENSG00000115350", "ENSG00000115355", "ENSG00000115364", "ENSG00000115365", "ENSG00000115368", "ENSG00000115380", "ENSG00000115415", "ENSG00000115419", "ENSG00000115421", "ENSG00000115446", "ENSG00000115464", "ENSG00000115468", "ENSG00000115484", "ENSG00000115504", "ENSG00000115514", "ENSG00000115520", "ENSG00000115524", "ENSG00000115525", "ENSG00000115526", "ENSG00000115548", "ENSG00000115561", "ENSG00000115568", "ENSG00000115649", "ENSG00000115652", "ENSG00000115685", "ENSG00000115694", "ENSG00000115756", "ENSG00000115758", "ENSG00000115760", "ENSG00000115762", "ENSG00000115806", "ENSG00000115808", "ENSG00000115827", "ENSG00000115828", "ENSG00000115839", "ENSG00000115840", "ENSG00000115875", "ENSG00000115884", "ENSG00000115902", "ENSG00000115904", "ENSG00000115944", "ENSG00000115947", "ENSG00000115966", "ENSG00000115977", "ENSG00000115993", "ENSG00000116001", "ENSG00000116005", "ENSG00000116016", "ENSG00000116044", "ENSG00000116095", "ENSG00000116120", "ENSG00000116128", "ENSG00000116133", "ENSG00000116138", "ENSG00000116141", "ENSG00000116147", "ENSG00000116161", "ENSG00000116171", "ENSG00000116199", "ENSG00000116237", "ENSG00000116251", "ENSG00000116254", "ENSG00000116260", "ENSG00000116266", "ENSG00000116285", "ENSG00000116288", "ENSG00000116337", "ENSG00000116350", "ENSG00000116396", "ENSG00000116406", "ENSG00000116455", "ENSG00000116473", "ENSG00000116489", "ENSG00000116497", "ENSG00000116521", "ENSG00000116539", "ENSG00000116544", "ENSG00000116560", "ENSG00000116604", "ENSG00000116641", "ENSG00000116649", "ENSG00000116661", "ENSG00000116667", "ENSG00000116675", "ENSG00000116678", "ENSG00000116679", "ENSG00000116685", "ENSG00000116688", "ENSG00000116698", "ENSG00000116717", "ENSG00000116731", "ENSG00000116741", "ENSG00000116750", "ENSG00000116752", "ENSG00000116754", "ENSG00000116791", "ENSG00000116793", "ENSG00000116809", "ENSG00000116852", "ENSG00000116857", "ENSG00000116871", "ENSG00000116898", "ENSG00000116903", "ENSG00000116906", "ENSG00000116918", "ENSG00000116977", "ENSG00000116983", "ENSG00000116991", "ENSG00000117016", "ENSG00000117020", "ENSG00000117054", "ENSG00000117069", "ENSG00000117114", "ENSG00000117115", "ENSG00000117118", "ENSG00000117133", "ENSG00000117139", "ENSG00000117143", "ENSG00000117152", "ENSG00000117153", "ENSG00000117155", "ENSG00000117174", "ENSG00000117222", "ENSG00000117280", "ENSG00000117335", "ENSG00000117394", "ENSG00000117395", "ENSG00000117408", "ENSG00000117410", "ENSG00000117411", "ENSG00000117448", "ENSG00000117450", "ENSG00000117461", "ENSG00000117475", "ENSG00000117477", "ENSG00000117479", "ENSG00000117481", "ENSG00000117500", "ENSG00000117505", "ENSG00000117519", "ENSG00000117523", "ENSG00000117528", "ENSG00000117533", "ENSG00000117569", "ENSG00000117592", "ENSG00000117602", "ENSG00000117614", "ENSG00000117616", "ENSG00000117625", "ENSG00000117632", "ENSG00000117640", "ENSG00000117643", "ENSG00000117676", "ENSG00000117682", "ENSG00000117691", "ENSG00000117697", "ENSG00000117713", "ENSG00000117748", "ENSG00000117751", "ENSG00000117758", "ENSG00000117859", "ENSG00000117868", "ENSG00000117906", "ENSG00000117984", "ENSG00000118046", "ENSG00000118058", "ENSG00000118096", "ENSG00000118160", "ENSG00000118200", "ENSG00000118217", "ENSG00000118260", "ENSG00000118263", "ENSG00000118276", "ENSG00000118369", "ENSG00000118402", "ENSG00000118418", "ENSG00000118454", "ENSG00000118473", "ENSG00000118482", "ENSG00000118496", "ENSG00000118507", "ENSG00000118515", "ENSG00000118518", "ENSG00000118564", "ENSG00000118579", "ENSG00000118680", "ENSG00000118705", "ENSG00000118733", "ENSG00000118785", "ENSG00000118816", "ENSG00000118855", "ENSG00000118873", "ENSG00000118946", "ENSG00000118965", "ENSG00000118985", "ENSG00000119013", "ENSG00000119041", "ENSG00000119048", "ENSG00000119138", "ENSG00000119185", "ENSG00000119203", "ENSG00000119227", "ENSG00000119231", "ENSG00000119242", "ENSG00000119280", "ENSG00000119314", "ENSG00000119318", "ENSG00000119335", "ENSG00000119396", "ENSG00000119401", "ENSG00000119402", "ENSG00000119414", "ENSG00000119421", "ENSG00000119446", "ENSG00000119471", "ENSG00000119487", "ENSG00000119522", "ENSG00000119523", "ENSG00000119537", "ENSG00000119541", "ENSG00000119559", "ENSG00000119574", "ENSG00000119596", "ENSG00000119632", "ENSG00000119638", "ENSG00000119640", "ENSG00000119650", "ENSG00000119661", "ENSG00000119669", "ENSG00000119682", "ENSG00000119698", "ENSG00000119705", "ENSG00000119707", "ENSG00000119711", "ENSG00000119718", "ENSG00000119729", "ENSG00000119760", "ENSG00000119771", "ENSG00000119772", "ENSG00000119777", "ENSG00000119787", "ENSG00000119801", "ENSG00000119812", "ENSG00000119820", "ENSG00000119844", "ENSG00000119862", "ENSG00000119865", "ENSG00000119866", "ENSG00000119878", "ENSG00000119899", "ENSG00000119900", "ENSG00000119906", "ENSG00000119938", "ENSG00000119946", "ENSG00000119950", "ENSG00000119953", "ENSG00000119965", "ENSG00000119977", "ENSG00000119986", "ENSG00000120008", "ENSG00000120053", "ENSG00000120063", "ENSG00000120129", "ENSG00000120137", "ENSG00000120162", "ENSG00000120251", "ENSG00000120265", "ENSG00000120306", "ENSG00000120333", "ENSG00000120438", "ENSG00000120451", "ENSG00000120509", "ENSG00000120533", "ENSG00000120594", "ENSG00000120616", "ENSG00000120645", "ENSG00000120658", "ENSG00000120675", "ENSG00000120685", "ENSG00000120686", "ENSG00000120688", "ENSG00000120693", "ENSG00000120694", "ENSG00000120696", "ENSG00000120697", "ENSG00000120705", "ENSG00000120709", "ENSG00000120727", "ENSG00000120733", "ENSG00000120742", "ENSG00000120798", "ENSG00000120800", "ENSG00000120802", "ENSG00000120805", "ENSG00000120832", "ENSG00000120837", "ENSG00000120885", "ENSG00000120913", "ENSG00000120925", "ENSG00000120948", "ENSG00000120992", "ENSG00000121022", "ENSG00000121057", "ENSG00000121064", "ENSG00000121067", "ENSG00000121073", "ENSG00000121310", "ENSG00000121390", "ENSG00000121413", "ENSG00000121486", "ENSG00000121579", "ENSG00000121671", "ENSG00000121680", "ENSG00000121741", "ENSG00000121749", "ENSG00000121753", "ENSG00000121766", "ENSG00000121769", "ENSG00000121774", "ENSG00000121871", "ENSG00000121879", "ENSG00000121892", "ENSG00000121900", "ENSG00000121904", "ENSG00000121964", "ENSG00000121989", "ENSG00000122012", "ENSG00000122034", "ENSG00000122042", "ENSG00000122068", "ENSG00000122085", "ENSG00000122126", "ENSG00000122203", "ENSG00000122218", "ENSG00000122257", "ENSG00000122299", "ENSG00000122335", "ENSG00000122417", "ENSG00000122484", "ENSG00000122515", "ENSG00000122550", "ENSG00000122565", "ENSG00000122566", "ENSG00000122574", "ENSG00000122584", "ENSG00000122692", "ENSG00000122705", "ENSG00000122729", "ENSG00000122733", "ENSG00000122741", "ENSG00000122778", "ENSG00000122779", "ENSG00000122786", "ENSG00000122873", "ENSG00000122882", "ENSG00000122912", "ENSG00000122958", "ENSG00000122965", "ENSG00000123066", "ENSG00000123091", "ENSG00000123095", "ENSG00000123106", "ENSG00000123124", "ENSG00000123159", "ENSG00000123178", "ENSG00000123200", "ENSG00000123213", "ENSG00000123240", "ENSG00000123352", "ENSG00000123384", "ENSG00000123472", "ENSG00000123505", "ENSG00000123545", "ENSG00000123560", "ENSG00000123570", "ENSG00000123575", "ENSG00000123595", "ENSG00000123636", "ENSG00000123684", "ENSG00000123728", "ENSG00000123739", "ENSG00000123836", "ENSG00000123983", "ENSG00000123989", "ENSG00000124006", "ENSG00000124098", "ENSG00000124104", "ENSG00000124120", "ENSG00000124126", "ENSG00000124140", "ENSG00000124145", "ENSG00000124151", "ENSG00000124155", "ENSG00000124160", "ENSG00000124164", "ENSG00000124177", "ENSG00000124181", "ENSG00000124191", "ENSG00000124193", "ENSG00000124194", "ENSG00000124198", "ENSG00000124207", "ENSG00000124209", "ENSG00000124214", "ENSG00000124222", "ENSG00000124225", "ENSG00000124226", "ENSG00000124228", "ENSG00000124243", "ENSG00000124275", "ENSG00000124299", "ENSG00000124356", "ENSG00000124380", "ENSG00000124383", "ENSG00000124406", "ENSG00000124422", "ENSG00000124444", "ENSG00000124486", "ENSG00000124507", "ENSG00000124523", "ENSG00000124532", "ENSG00000124535", "ENSG00000124541", "ENSG00000124570", "ENSG00000124571", "ENSG00000124588", "ENSG00000124596", "ENSG00000124702", "ENSG00000124733", "ENSG00000124766", "ENSG00000124767", "ENSG00000124783", "ENSG00000124784", "ENSG00000124785", "ENSG00000124788", "ENSG00000124789", "ENSG00000124795", "ENSG00000124802", "ENSG00000124831", "ENSG00000125107", "ENSG00000125124", "ENSG00000125144", "ENSG00000125148", "ENSG00000125166", "ENSG00000125170", "ENSG00000125247", "ENSG00000125249", "ENSG00000125304", "ENSG00000125351", "ENSG00000125354", "ENSG00000125355", "ENSG00000125356", "ENSG00000125447", "ENSG00000125484", "ENSG00000125503", "ENSG00000125505", "ENSG00000125510", "ENSG00000125534", "ENSG00000125629", "ENSG00000125633", "ENSG00000125648", "ENSG00000125656", "ENSG00000125675", "ENSG00000125676", "ENSG00000125730", "ENSG00000125741", "ENSG00000125743", "ENSG00000125744", "ENSG00000125746", "ENSG00000125772", "ENSG00000125779", "ENSG00000125798", "ENSG00000125812", "ENSG00000125814", "ENSG00000125818", "ENSG00000125826", "ENSG00000125827", "ENSG00000125834", "ENSG00000125843", "ENSG00000125851", "ENSG00000125863", "ENSG00000125868", "ENSG00000125870", "ENSG00000125875", "ENSG00000125901", "ENSG00000125944", "ENSG00000125952", "ENSG00000125962", "ENSG00000125967", "ENSG00000125977", "ENSG00000125991", "ENSG00000125995", "ENSG00000126012", "ENSG00000126067", "ENSG00000126070", "ENSG00000126088", "ENSG00000126107", "ENSG00000126214", "ENSG00000126216", "ENSG00000126217", "ENSG00000126247", "ENSG00000126254", "ENSG00000126267", "ENSG00000126351", "ENSG00000126432", "ENSG00000126653", "ENSG00000126698", "ENSG00000126705", "ENSG00000126733", "ENSG00000126749", "ENSG00000126756", "ENSG00000126768", "ENSG00000126777", "ENSG00000126803", "ENSG00000126804", "ENSG00000126821", "ENSG00000126858", "ENSG00000126861", "ENSG00000126883", "ENSG00000126934", "ENSG00000126945", "ENSG00000126947", "ENSG00000126970", "ENSG00000127022", "ENSG00000127124", "ENSG00000127125", "ENSG00000127184", "ENSG00000127328", "ENSG00000127334", "ENSG00000127419", "ENSG00000127445", "ENSG00000127463", "ENSG00000127483", "ENSG00000127511", "ENSG00000127526", "ENSG00000127527", "ENSG00000127540", "ENSG00000127561", "ENSG00000127585", "ENSG00000127603", "ENSG00000127616", "ENSG00000127824", "ENSG00000127837", "ENSG00000127838", "ENSG00000127870", "ENSG00000127884", "ENSG00000127914", "ENSG00000127922", "ENSG00000127947", "ENSG00000127952", "ENSG00000127955", "ENSG00000127980", "ENSG00000127990", "ENSG00000127995", "ENSG00000128000", "ENSG00000128011", "ENSG00000128050", "ENSG00000128059", "ENSG00000128203", "ENSG00000128245", "ENSG00000128266", "ENSG00000128268", "ENSG00000128335", "ENSG00000128463", "ENSG00000128482", "ENSG00000128487", "ENSG00000128512", "ENSG00000128524", "ENSG00000128563", "ENSG00000128567", "ENSG00000128573", "ENSG00000128578", "ENSG00000128581", "ENSG00000128585", "ENSG00000128590", "ENSG00000128595", "ENSG00000128596", "ENSG00000128607", "ENSG00000128654", "ENSG00000128655", "ENSG00000128656", "ENSG00000128699", "ENSG00000128731", "ENSG00000128789", "ENSG00000128791", "ENSG00000128829", "ENSG00000128872", "ENSG00000128881", "ENSG00000128928", "ENSG00000128951", "ENSG00000128989", "ENSG00000129003", "ENSG00000129055", "ENSG00000129071", "ENSG00000129083", "ENSG00000129103", "ENSG00000129116", "ENSG00000129128", "ENSG00000129158", "ENSG00000129187", "ENSG00000129194", "ENSG00000129235", "ENSG00000129244", "ENSG00000129245", "ENSG00000129250", "ENSG00000129292", "ENSG00000129315", "ENSG00000129347", "ENSG00000129351", "ENSG00000129353", "ENSG00000129422", "ENSG00000129460", "ENSG00000129472", "ENSG00000129480", "ENSG00000129484", "ENSG00000129493", "ENSG00000129515", "ENSG00000129518", "ENSG00000129559", "ENSG00000129562", "ENSG00000129596", "ENSG00000129625", "ENSG00000129636", "ENSG00000129657", "ENSG00000129675", "ENSG00000129682", "ENSG00000129691", "ENSG00000129757", "ENSG00000129824", "ENSG00000129932", "ENSG00000129933", "ENSG00000129946", "ENSG00000130024", "ENSG00000130032", "ENSG00000130066", "ENSG00000130159", "ENSG00000130165", "ENSG00000130175", "ENSG00000130177", "ENSG00000130204", "ENSG00000130208", "ENSG00000130224", "ENSG00000130226", "ENSG00000130254", "ENSG00000130255", "ENSG00000130287", "ENSG00000130294", "ENSG00000130299", "ENSG00000130311", "ENSG00000130313", "ENSG00000130338", "ENSG00000130340", "ENSG00000130363", "ENSG00000130402", "ENSG00000130414", "ENSG00000130449", "ENSG00000130477", "ENSG00000130517", "ENSG00000130520", "ENSG00000130522", "ENSG00000130540", "ENSG00000130558", "ENSG00000130559", "ENSG00000130560", "ENSG00000130638", "ENSG00000130643", "ENSG00000130703", "ENSG00000130707", "ENSG00000130714", "ENSG00000130717", "ENSG00000130724", "ENSG00000130733", "ENSG00000130741", "ENSG00000130749", "ENSG00000130764", "ENSG00000130766", "ENSG00000130779", "ENSG00000130816", "ENSG00000130821", "ENSG00000130826", "ENSG00000130827", "ENSG00000130830", "ENSG00000130844", "ENSG00000130939", "ENSG00000130956", "ENSG00000130985", "ENSG00000131013", "ENSG00000131016", "ENSG00000131018", "ENSG00000131051", "ENSG00000131069", "ENSG00000131089", "ENSG00000131095", "ENSG00000131100", "ENSG00000131116", "ENSG00000131143", "ENSG00000131148", "ENSG00000131149", "ENSG00000131165", "ENSG00000131171", "ENSG00000131174", "ENSG00000131236", "ENSG00000131238", "ENSG00000131242", "ENSG00000131263", "ENSG00000131323", "ENSG00000131368", "ENSG00000131374", "ENSG00000131375", "ENSG00000131378", "ENSG00000131381", "ENSG00000131437", "ENSG00000131446", "ENSG00000131467", "ENSG00000131469", "ENSG00000131473", "ENSG00000131495", "ENSG00000131504", "ENSG00000131507", "ENSG00000131508", "ENSG00000131558", "ENSG00000131626", "ENSG00000131711", "ENSG00000131725", "ENSG00000131732", "ENSG00000131773", "ENSG00000131779", "ENSG00000131791", "ENSG00000131828", "ENSG00000131831", "ENSG00000131943", "ENSG00000131966", "ENSG00000131979", "ENSG00000132002", "ENSG00000132024", "ENSG00000132031", "ENSG00000132128", "ENSG00000132153", "ENSG00000132155", "ENSG00000132164", "ENSG00000132182", "ENSG00000132275", "ENSG00000132294", "ENSG00000132300", "ENSG00000132305", "ENSG00000132313", "ENSG00000132321", "ENSG00000132329", "ENSG00000132334", "ENSG00000132341", "ENSG00000132356", "ENSG00000132359", "ENSG00000132361", "ENSG00000132383", "ENSG00000132388", "ENSG00000132424", "ENSG00000132429", "ENSG00000132432", "ENSG00000132434", "ENSG00000132437", "ENSG00000132463", "ENSG00000132466", "ENSG00000132467", "ENSG00000132471", "ENSG00000132485", "ENSG00000132507", "ENSG00000132535", "ENSG00000132561", "ENSG00000132563", "ENSG00000132591", "ENSG00000132604", "ENSG00000132612", "ENSG00000132639", "ENSG00000132640", "ENSG00000132664", "ENSG00000132670", "ENSG00000132676", "ENSG00000132718", "ENSG00000132740", "ENSG00000132792", "ENSG00000132821", "ENSG00000132823", "ENSG00000132824", "ENSG00000132842", "ENSG00000132854", "ENSG00000132872", "ENSG00000132879", "ENSG00000132912", "ENSG00000132932", "ENSG00000132953", "ENSG00000132963", "ENSG00000132964", "ENSG00000132970", "ENSG00000132975", "ENSG00000133026", "ENSG00000133027", "ENSG00000133030", "ENSG00000133048", "ENSG00000133056", "ENSG00000133059", "ENSG00000133065", "ENSG00000133069", "ENSG00000133083", "ENSG00000133101", "ENSG00000133103", "ENSG00000133104", "ENSG00000133112", "ENSG00000133114", "ENSG00000133134", "ENSG00000133135", "ENSG00000133142", "ENSG00000133169", "ENSG00000133216", "ENSG00000133226", "ENSG00000133243", "ENSG00000133275", "ENSG00000133313", "ENSG00000133398", "ENSG00000133401", "ENSG00000133424", "ENSG00000133460", "ENSG00000133597", "ENSG00000133606", "ENSG00000133612", "ENSG00000133627", "ENSG00000133657", "ENSG00000133687", "ENSG00000133703", "ENSG00000133704", "ENSG00000133731", "ENSG00000133805", "ENSG00000133812", "ENSG00000133835", "ENSG00000133858", "ENSG00000133872", "ENSG00000133878", "ENSG00000133884", "ENSG00000133935", "ENSG00000133961", "ENSG00000133985", "ENSG00000133997", "ENSG00000134001", "ENSG00000134014", "ENSG00000134030", "ENSG00000134046", "ENSG00000134049", "ENSG00000134107", "ENSG00000134108", "ENSG00000134115", "ENSG00000134121", "ENSG00000134152", "ENSG00000134153", "ENSG00000134186", "ENSG00000134202", "ENSG00000134215", "ENSG00000134243", "ENSG00000134248", "ENSG00000134262", "ENSG00000134265", "ENSG00000134283", "ENSG00000134287", "ENSG00000134291", "ENSG00000134294", "ENSG00000134308", "ENSG00000134313", "ENSG00000134318", "ENSG00000134324", "ENSG00000134330", "ENSG00000134352", "ENSG00000134369", "ENSG00000134371", "ENSG00000134375", "ENSG00000134453", "ENSG00000134480", "ENSG00000134504", "ENSG00000134533", "ENSG00000134644", "ENSG00000134684", "ENSG00000134686", "ENSG00000134709", "ENSG00000134717", "ENSG00000134748", "ENSG00000134758", "ENSG00000134759", "ENSG00000134769", "ENSG00000134779", "ENSG00000134809", "ENSG00000134825", "ENSG00000134851", "ENSG00000134852", "ENSG00000134874", "ENSG00000134884", "ENSG00000134897", "ENSG00000134900", "ENSG00000134905", "ENSG00000134909", "ENSG00000134970", "ENSG00000134982", "ENSG00000134986", "ENSG00000134987", "ENSG00000134996", "ENSG00000135002", "ENSG00000135018", "ENSG00000135040", "ENSG00000135045", "ENSG00000135047", "ENSG00000135049", "ENSG00000135052", "ENSG00000135069", "ENSG00000135070", "ENSG00000135090", "ENSG00000135108", "ENSG00000135119", "ENSG00000135144", "ENSG00000135164", "ENSG00000135211", "ENSG00000135241", "ENSG00000135250", "ENSG00000135298", "ENSG00000135299", "ENSG00000135333", "ENSG00000135334", "ENSG00000135336", "ENSG00000135341", "ENSG00000135365", "ENSG00000135372", "ENSG00000135387", "ENSG00000135404", "ENSG00000135414", "ENSG00000135447", "ENSG00000135454", "ENSG00000135457", "ENSG00000135469", "ENSG00000135472", "ENSG00000135486", "ENSG00000135503", "ENSG00000135525", "ENSG00000135535", "ENSG00000135541", "ENSG00000135549", "ENSG00000135597", "ENSG00000135617", "ENSG00000135624", "ENSG00000135631", "ENSG00000135636", "ENSG00000135655", "ENSG00000135677", "ENSG00000135679", "ENSG00000135686", "ENSG00000135698", "ENSG00000135709", "ENSG00000135744", "ENSG00000135778", "ENSG00000135821", "ENSG00000135823", "ENSG00000135829", "ENSG00000135837", "ENSG00000135838", "ENSG00000135845", "ENSG00000135862", "ENSG00000135870", "ENSG00000135905", "ENSG00000135916", "ENSG00000135919", "ENSG00000135924", "ENSG00000135926", "ENSG00000135930", "ENSG00000135932", "ENSG00000135940", "ENSG00000135956", "ENSG00000135966", "ENSG00000135968", "ENSG00000135972", "ENSG00000135974", "ENSG00000136003", "ENSG00000136021", "ENSG00000136026", "ENSG00000136040", "ENSG00000136045", "ENSG00000136068", "ENSG00000136099", "ENSG00000136100", "ENSG00000136111", "ENSG00000136143", "ENSG00000136144", "ENSG00000136146", "ENSG00000136156", "ENSG00000136159", "ENSG00000136160", "ENSG00000136193", "ENSG00000136235", "ENSG00000136237", "ENSG00000136240", "ENSG00000136261", "ENSG00000136271", "ENSG00000136273", "ENSG00000136274", "ENSG00000136280", "ENSG00000136295", "ENSG00000136319", "ENSG00000136367", "ENSG00000136381", "ENSG00000136383", "ENSG00000136436", "ENSG00000136444", "ENSG00000136448", "ENSG00000136450", "ENSG00000136451", "ENSG00000136478", "ENSG00000136485", "ENSG00000136504", "ENSG00000136521", "ENSG00000136527", "ENSG00000136531", "ENSG00000136541", "ENSG00000136546", "ENSG00000136560", "ENSG00000136631", "ENSG00000136636", "ENSG00000136643", "ENSG00000136709", "ENSG00000136717", "ENSG00000136718", "ENSG00000136720", "ENSG00000136731", "ENSG00000136738", "ENSG00000136754", "ENSG00000136758", "ENSG00000136783", "ENSG00000136802", "ENSG00000136807", "ENSG00000136810", "ENSG00000136816", "ENSG00000136819", "ENSG00000136824", "ENSG00000136827", "ENSG00000136828", "ENSG00000136848", "ENSG00000136854", "ENSG00000136856", "ENSG00000136870", "ENSG00000136875", "ENSG00000136878", "ENSG00000136888", "ENSG00000136891", "ENSG00000136895", "ENSG00000136897", "ENSG00000136928", "ENSG00000136930", "ENSG00000136933", "ENSG00000136936", "ENSG00000136937", "ENSG00000136938", "ENSG00000136940", "ENSG00000136942", "ENSG00000136950", "ENSG00000136960", "ENSG00000136986", "ENSG00000137040", "ENSG00000137055", "ENSG00000137073", "ENSG00000137074", "ENSG00000137075", "ENSG00000137100", "ENSG00000137103", "ENSG00000137106", "ENSG00000137166", "ENSG00000137168", "ENSG00000137171", "ENSG00000137177", "ENSG00000137193", "ENSG00000137198", "ENSG00000137200", "ENSG00000137210", "ENSG00000137216", "ENSG00000137218", "ENSG00000137261", "ENSG00000137266", "ENSG00000137267", "ENSG00000137285", "ENSG00000137288", "ENSG00000137309", "ENSG00000137364", "ENSG00000137393", "ENSG00000137409", "ENSG00000137414", "ENSG00000137449", "ENSG00000137478", "ENSG00000137486", "ENSG00000137500", "ENSG00000137501", "ENSG00000137502", "ENSG00000137504", "ENSG00000137509", "ENSG00000137513", "ENSG00000137547", "ENSG00000137563", "ENSG00000137575", "ENSG00000137601", "ENSG00000137642", "ENSG00000137692", "ENSG00000137710", "ENSG00000137714", "ENSG00000137726", "ENSG00000137766", "ENSG00000137776", "ENSG00000137806", "ENSG00000137815", "ENSG00000137817", "ENSG00000137821", "ENSG00000137822", "ENSG00000137824", "ENSG00000137845", "ENSG00000137871", "ENSG00000137872", "ENSG00000137876", "ENSG00000137941", "ENSG00000137942", "ENSG00000137947", "ENSG00000137955", "ENSG00000137968", "ENSG00000137992", "ENSG00000137996", "ENSG00000138029", "ENSG00000138031", "ENSG00000138032", "ENSG00000138035", "ENSG00000138036", "ENSG00000138069", "ENSG00000138071", "ENSG00000138073", "ENSG00000138074", "ENSG00000138078", "ENSG00000138085", "ENSG00000138095", "ENSG00000138101", "ENSG00000138107", "ENSG00000138138", "ENSG00000138162", "ENSG00000138175", "ENSG00000138190", "ENSG00000138279", "ENSG00000138303", "ENSG00000138311", "ENSG00000138326", "ENSG00000138363", "ENSG00000138381", "ENSG00000138385", "ENSG00000138386", "ENSG00000138398", "ENSG00000138399", "ENSG00000138411", "ENSG00000138413", "ENSG00000138430", "ENSG00000138433", "ENSG00000138439", "ENSG00000138442", "ENSG00000138443", "ENSG00000138448", "ENSG00000138449", "ENSG00000138459", "ENSG00000138468", "ENSG00000138592", "ENSG00000138593", "ENSG00000138604", "ENSG00000138613", "ENSG00000138622", "ENSG00000138639", "ENSG00000138642", "ENSG00000138650", "ENSG00000138663", "ENSG00000138668", "ENSG00000138674", "ENSG00000138686", "ENSG00000138698", "ENSG00000138709", "ENSG00000138744", "ENSG00000138757", "ENSG00000138760", "ENSG00000138764", "ENSG00000138768", "ENSG00000138777", "ENSG00000138785", "ENSG00000138796", "ENSG00000138801", "ENSG00000138802", "ENSG00000138814", "ENSG00000138835", "ENSG00000138942", "ENSG00000139112", "ENSG00000139116", "ENSG00000139132", "ENSG00000139163", "ENSG00000139168", "ENSG00000139174", "ENSG00000139180", "ENSG00000139182", "ENSG00000139190", "ENSG00000139197", "ENSG00000139211", "ENSG00000139218", "ENSG00000139220", "ENSG00000139291", "ENSG00000139324", "ENSG00000139343", "ENSG00000139364", "ENSG00000139405", "ENSG00000139436", "ENSG00000139438", "ENSG00000139514", "ENSG00000139531", "ENSG00000139624", "ENSG00000139641", "ENSG00000139644", "ENSG00000139645", "ENSG00000139684", "ENSG00000139687", "ENSG00000139697", "ENSG00000139718", "ENSG00000139719", "ENSG00000139726", "ENSG00000139737", "ENSG00000139746", "ENSG00000139767", "ENSG00000139793", "ENSG00000139842", "ENSG00000139874", "ENSG00000139910", "ENSG00000139921", "ENSG00000139970", "ENSG00000139977", "ENSG00000139990", "ENSG00000139998", "ENSG00000140044", "ENSG00000140259", "ENSG00000140262", "ENSG00000140280", "ENSG00000140299", "ENSG00000140307", "ENSG00000140319", "ENSG00000140320", "ENSG00000140332", "ENSG00000140350", "ENSG00000140365", "ENSG00000140367", "ENSG00000140374", "ENSG00000140382", "ENSG00000140391", "ENSG00000140396", "ENSG00000140403", "ENSG00000140416", "ENSG00000140443", "ENSG00000140463", "ENSG00000140474", "ENSG00000140526", "ENSG00000140538", "ENSG00000140553", "ENSG00000140564", "ENSG00000140575", "ENSG00000140577", "ENSG00000140600", "ENSG00000140612", "ENSG00000140691", "ENSG00000140694", "ENSG00000140718", "ENSG00000140740", "ENSG00000140836", "ENSG00000140854", "ENSG00000140937", "ENSG00000140939", "ENSG00000140941", "ENSG00000140943", "ENSG00000140983", "ENSG00000140990", "ENSG00000140992", "ENSG00000140995", "ENSG00000141002", "ENSG00000141026", "ENSG00000141027", "ENSG00000141030", "ENSG00000141098", "ENSG00000141127", "ENSG00000141232", "ENSG00000141252", "ENSG00000141258", "ENSG00000141279", "ENSG00000141298", "ENSG00000141338", "ENSG00000141349", "ENSG00000141367", "ENSG00000141378", "ENSG00000141384", "ENSG00000141385", "ENSG00000141404", "ENSG00000141424", "ENSG00000141425", "ENSG00000141429", "ENSG00000141431", "ENSG00000141447", "ENSG00000141456", "ENSG00000141458", "ENSG00000141469", "ENSG00000141503", "ENSG00000141522", "ENSG00000141542", "ENSG00000141543", "ENSG00000141551", "ENSG00000141556", "ENSG00000141560", "ENSG00000141562", "ENSG00000141568", "ENSG00000141576", "ENSG00000141580", "ENSG00000141582", "ENSG00000141644", "ENSG00000141664", "ENSG00000141668", "ENSG00000141741", "ENSG00000141759", "ENSG00000141837", "ENSG00000141867", "ENSG00000141905", "ENSG00000141959", "ENSG00000141985", "ENSG00000142046", "ENSG00000142082", "ENSG00000142089", "ENSG00000142166", "ENSG00000142168", "ENSG00000142186", "ENSG00000142188", "ENSG00000142192", "ENSG00000142207", "ENSG00000142230", "ENSG00000142235", "ENSG00000142252", "ENSG00000142319", "ENSG00000142408", "ENSG00000142453", "ENSG00000142507", "ENSG00000142534", "ENSG00000142546", "ENSG00000142599", "ENSG00000142623", "ENSG00000142634", "ENSG00000142655", "ENSG00000142669", "ENSG00000142676", "ENSG00000142687", "ENSG00000142784", "ENSG00000142864", "ENSG00000142875", "ENSG00000142892", "ENSG00000142949", "ENSG00000143013", "ENSG00000143033", "ENSG00000143061", "ENSG00000143079", "ENSG00000143093", "ENSG00000143106", "ENSG00000143126", "ENSG00000143147", "ENSG00000143153", "ENSG00000143155", "ENSG00000143157", "ENSG00000143158", "ENSG00000143162", "ENSG00000143164", "ENSG00000143179", "ENSG00000143183", "ENSG00000143190", "ENSG00000143198", "ENSG00000143222", "ENSG00000143224", "ENSG00000143248", "ENSG00000143256", "ENSG00000143294", "ENSG00000143314", "ENSG00000143315", "ENSG00000143321", "ENSG00000143322", "ENSG00000143324", "ENSG00000143333", "ENSG00000143337", "ENSG00000143344", "ENSG00000143353", "ENSG00000143376", "ENSG00000143384", "ENSG00000143393", "ENSG00000143401", "ENSG00000143420", "ENSG00000143437", "ENSG00000143442", "ENSG00000143457", "ENSG00000143493", "ENSG00000143502", "ENSG00000143515", "ENSG00000143543", "ENSG00000143549", "ENSG00000143553", "ENSG00000143569", "ENSG00000143575", "ENSG00000143603", "ENSG00000143612", "ENSG00000143621", "ENSG00000143630", "ENSG00000143641", "ENSG00000143653", "ENSG00000143669", "ENSG00000143702", "ENSG00000143727", "ENSG00000143740", "ENSG00000143751", "ENSG00000143753", "ENSG00000143756", "ENSG00000143761", "ENSG00000143771", "ENSG00000143772", "ENSG00000143774", "ENSG00000143776", "ENSG00000143786", "ENSG00000143793", "ENSG00000143797", "ENSG00000143799", "ENSG00000143811", "ENSG00000143847", "ENSG00000143850", "ENSG00000143862", "ENSG00000143878", "ENSG00000143889", "ENSG00000143919", "ENSG00000143924", "ENSG00000143933", "ENSG00000143970", "ENSG00000144021", "ENSG00000144028", "ENSG00000144029", "ENSG00000144036", "ENSG00000144040", "ENSG00000144136", "ENSG00000144161", "ENSG00000144218", "ENSG00000144224", "ENSG00000144228", "ENSG00000144283", "ENSG00000144306", "ENSG00000144339", "ENSG00000144357", "ENSG00000144369", "ENSG00000144401", "ENSG00000144406", "ENSG00000144445", "ENSG00000144451", "ENSG00000144524", "ENSG00000144566", "ENSG00000144580", "ENSG00000144597", "ENSG00000144619", "ENSG00000144635", "ENSG00000144642", "ENSG00000144645", "ENSG00000144647", "ENSG00000144659", "ENSG00000144674", "ENSG00000144677", "ENSG00000144711", "ENSG00000144724", "ENSG00000144741", "ENSG00000144744", "ENSG00000144746", "ENSG00000144747", "ENSG00000144815", "ENSG00000144827", "ENSG00000144834", "ENSG00000144840", "ENSG00000144848", "ENSG00000144867", "ENSG00000144868", "ENSG00000144891", "ENSG00000144895", "ENSG00000144935", "ENSG00000144959", "ENSG00000145012", "ENSG00000145022", "ENSG00000145050", "ENSG00000145087", "ENSG00000145147", "ENSG00000145191", "ENSG00000145242", "ENSG00000145246", "ENSG00000145248", "ENSG00000145284", "ENSG00000145293", "ENSG00000145332", "ENSG00000145335", "ENSG00000145348", "ENSG00000145349", "ENSG00000145362", "ENSG00000145388", "ENSG00000145390", "ENSG00000145391", "ENSG00000145428", "ENSG00000145439", "ENSG00000145476", "ENSG00000145494", "ENSG00000145545", "ENSG00000145555", "ENSG00000145592", "ENSG00000145675", "ENSG00000145687", "ENSG00000145725", "ENSG00000145730", "ENSG00000145734", "ENSG00000145740", "ENSG00000145741", "ENSG00000145781", "ENSG00000145782", "ENSG00000145817", "ENSG00000145819", "ENSG00000145824", "ENSG00000145833", "ENSG00000145860", "ENSG00000145864", "ENSG00000145868", "ENSG00000145882", "ENSG00000145901", "ENSG00000145907", "ENSG00000145916", "ENSG00000145919", "ENSG00000145920", "ENSG00000145934", "ENSG00000145979", "ENSG00000145990", "ENSG00000146005", "ENSG00000146006", "ENSG00000146007", "ENSG00000146021", "ENSG00000146067", "ENSG00000146072", "ENSG00000146083", "ENSG00000146085", "ENSG00000146151", "ENSG00000146223", "ENSG00000146242", "ENSG00000146247", "ENSG00000146267", "ENSG00000146281", "ENSG00000146416", "ENSG00000146425", "ENSG00000146433", "ENSG00000146457", "ENSG00000146463", "ENSG00000146476", "ENSG00000146535", "ENSG00000146540", "ENSG00000146587", "ENSG00000146676", "ENSG00000146701", "ENSG00000146731", "ENSG00000146842", "ENSG00000146938", "ENSG00000146966", "ENSG00000147010", "ENSG00000147027", "ENSG00000147044", "ENSG00000147099", "ENSG00000147123", "ENSG00000147130", "ENSG00000147133", "ENSG00000147155", "ENSG00000147162", "ENSG00000147164", "ENSG00000147224", "ENSG00000147251", "ENSG00000147274", "ENSG00000147324", "ENSG00000147400", "ENSG00000147408", "ENSG00000147416", "ENSG00000147419", "ENSG00000147421", "ENSG00000147432", "ENSG00000147454", "ENSG00000147457", "ENSG00000147475", "ENSG00000147481", "ENSG00000147526", "ENSG00000147533", "ENSG00000147588", "ENSG00000147642", "ENSG00000147649", "ENSG00000147650", "ENSG00000147654", "ENSG00000147669", "ENSG00000147677", "ENSG00000147684", "ENSG00000147853", "ENSG00000147854", "ENSG00000147862", "ENSG00000147894", "ENSG00000147955", "ENSG00000148019", "ENSG00000148053", "ENSG00000148090", "ENSG00000148143", "ENSG00000148153", "ENSG00000148154", "ENSG00000148158", "ENSG00000148175", "ENSG00000148180", "ENSG00000148218", "ENSG00000148219", "ENSG00000148229", "ENSG00000148248", "ENSG00000148290", "ENSG00000148296", "ENSG00000148300", "ENSG00000148308", "ENSG00000148334", "ENSG00000148335", "ENSG00000148337", "ENSG00000148341", "ENSG00000148358", "ENSG00000148384", "ENSG00000148396", "ENSG00000148408", "ENSG00000148411", "ENSG00000148444", "ENSG00000148450", "ENSG00000148468", "ENSG00000148484", "ENSG00000148498", "ENSG00000148516", "ENSG00000148541", "ENSG00000148572", "ENSG00000148606", "ENSG00000148634", "ENSG00000148660", "ENSG00000148672", "ENSG00000148688", "ENSG00000148690", "ENSG00000148700", "ENSG00000148719", "ENSG00000148730", "ENSG00000148798", "ENSG00000148834", "ENSG00000148840", "ENSG00000148842", "ENSG00000148843", "ENSG00000148925", "ENSG00000148926", "ENSG00000148943", "ENSG00000148948", "ENSG00000149084", "ENSG00000149100", "ENSG00000149136", "ENSG00000149182", "ENSG00000149187", "ENSG00000149212", "ENSG00000149218", "ENSG00000149231", "ENSG00000149256", "ENSG00000149269", "ENSG00000149273", "ENSG00000149294", "ENSG00000149295", "ENSG00000149308", "ENSG00000149311", "ENSG00000149313", "ENSG00000149357", "ENSG00000149428", "ENSG00000149485", "ENSG00000149532", "ENSG00000149541", "ENSG00000149547", "ENSG00000149557", "ENSG00000149571", "ENSG00000149575", "ENSG00000149577", "ENSG00000149600", "ENSG00000149657", "ENSG00000149658", "ENSG00000149743", "ENSG00000149792", "ENSG00000149970", "ENSG00000150316", "ENSG00000150347", "ENSG00000150361", "ENSG00000150394", "ENSG00000150401", "ENSG00000150403", "ENSG00000150459", "ENSG00000150471", "ENSG00000150510", "ENSG00000150540", "ENSG00000150593", "ENSG00000150625", "ENSG00000150627", "ENSG00000150656", "ENSG00000150712", "ENSG00000150753", "ENSG00000150768", "ENSG00000150787", "ENSG00000150867", "ENSG00000150938", "ENSG00000150967", "ENSG00000150991", "ENSG00000150995", "ENSG00000151012", "ENSG00000151025", "ENSG00000151090", "ENSG00000151092", "ENSG00000151116", "ENSG00000151135", "ENSG00000151148", "ENSG00000151150", "ENSG00000151176", "ENSG00000151229", "ENSG00000151240", "ENSG00000151247", "ENSG00000151276", "ENSG00000151292", "ENSG00000151320", "ENSG00000151327", "ENSG00000151332", "ENSG00000151348", "ENSG00000151353", "ENSG00000151366", "ENSG00000151376", "ENSG00000151414", "ENSG00000151445", "ENSG00000151458", "ENSG00000151461", "ENSG00000151474", "ENSG00000151490", "ENSG00000151491", "ENSG00000151498", "ENSG00000151500", "ENSG00000151502", "ENSG00000151552", "ENSG00000151612", "ENSG00000151640", "ENSG00000151689", "ENSG00000151690", "ENSG00000151692", "ENSG00000151693", "ENSG00000151726", "ENSG00000151729", "ENSG00000151746", "ENSG00000151748", "ENSG00000151778", "ENSG00000151779", "ENSG00000151789", "ENSG00000151806", "ENSG00000151835", "ENSG00000151849", "ENSG00000151883", "ENSG00000151892", "ENSG00000151893", "ENSG00000151914", "ENSG00000151917", "ENSG00000151923", "ENSG00000151929", "ENSG00000152061", "ENSG00000152092", "ENSG00000152102", "ENSG00000152127", "ENSG00000152133", "ENSG00000152137", "ENSG00000152154", "ENSG00000152223", "ENSG00000152291", "ENSG00000152332", "ENSG00000152377", "ENSG00000152404", "ENSG00000152409", "ENSG00000152413", "ENSG00000152465", "ENSG00000152484", "ENSG00000152492", "ENSG00000152503", "ENSG00000152518", "ENSG00000152556", "ENSG00000152578", "ENSG00000152583", "ENSG00000152601", "ENSG00000152620", "ENSG00000152642", "ENSG00000152661", "ENSG00000152684", "ENSG00000152700", "ENSG00000152749", "ENSG00000152767", "ENSG00000152778", "ENSG00000152795", "ENSG00000152818", "ENSG00000152822", "ENSG00000152904", "ENSG00000152926", "ENSG00000152932", "ENSG00000152944", "ENSG00000152954", "ENSG00000153130", "ENSG00000153132", "ENSG00000153140", "ENSG00000153187", "ENSG00000153201", "ENSG00000153214", "ENSG00000153234", "ENSG00000153253", "ENSG00000153291", "ENSG00000153317", "ENSG00000153339", "ENSG00000153395", "ENSG00000153406", "ENSG00000153558", "ENSG00000153560", "ENSG00000153561", "ENSG00000153707", "ENSG00000153774", "ENSG00000153786", "ENSG00000153814", "ENSG00000153815", "ENSG00000153822", "ENSG00000153823", "ENSG00000153827", "ENSG00000153879", "ENSG00000153904", "ENSG00000153914", "ENSG00000153933", "ENSG00000153936", "ENSG00000153944", "ENSG00000153956", "ENSG00000153982", "ENSG00000153989", "ENSG00000154001", "ENSG00000154027", "ENSG00000154059", "ENSG00000154065", "ENSG00000154096", "ENSG00000154122", "ENSG00000154127", "ENSG00000154144", "ENSG00000154217", "ENSG00000154222", "ENSG00000154229", "ENSG00000154265", "ENSG00000154277", "ENSG00000154305", "ENSG00000154359", "ENSG00000154370", "ENSG00000154380", "ENSG00000154429", "ENSG00000154473", "ENSG00000154478", "ENSG00000154556", "ENSG00000154639", "ENSG00000154723", "ENSG00000154803", "ENSG00000154813", "ENSG00000154822", "ENSG00000154845", "ENSG00000154917", "ENSG00000154945", "ENSG00000154978", "ENSG00000155016", "ENSG00000155085", "ENSG00000155093", "ENSG00000155096", "ENSG00000155097", "ENSG00000155100", "ENSG00000155158", "ENSG00000155189", "ENSG00000155229", "ENSG00000155252", "ENSG00000155287", "ENSG00000155304", "ENSG00000155313", "ENSG00000155329", "ENSG00000155368", "ENSG00000155380", "ENSG00000155463", "ENSG00000155506", "ENSG00000155511", "ENSG00000155657", "ENSG00000155660", "ENSG00000155755", "ENSG00000155792", "ENSG00000155816", "ENSG00000155827", "ENSG00000155846", "ENSG00000155849", "ENSG00000155858", "ENSG00000155868", "ENSG00000155876", "ENSG00000155886", "ENSG00000155903", "ENSG00000155957", "ENSG00000155959", "ENSG00000155961", "ENSG00000155966", "ENSG00000155970", "ENSG00000155975", "ENSG00000155980", "ENSG00000156011", "ENSG00000156050", "ENSG00000156052", "ENSG00000156103", "ENSG00000156110", "ENSG00000156113", "ENSG00000156136", "ENSG00000156162", "ENSG00000156170", "ENSG00000156239", "ENSG00000156253", "ENSG00000156256", "ENSG00000156261", "ENSG00000156298", "ENSG00000156304", "ENSG00000156345", "ENSG00000156467", "ENSG00000156471", "ENSG00000156475", "ENSG00000156482", "ENSG00000156515", "ENSG00000156531", "ENSG00000156587", "ENSG00000156599", "ENSG00000156603", "ENSG00000156639", "ENSG00000156642", "ENSG00000156650", "ENSG00000156709", "ENSG00000156735", "ENSG00000156873", "ENSG00000156928", "ENSG00000156931", "ENSG00000156990", "ENSG00000157014", "ENSG00000157020", "ENSG00000157036", "ENSG00000157045", "ENSG00000157064", "ENSG00000157087", "ENSG00000157103", "ENSG00000157106", "ENSG00000157152", "ENSG00000157193", "ENSG00000157224", "ENSG00000157259", "ENSG00000157445", "ENSG00000157450", "ENSG00000157500", "ENSG00000157514", "ENSG00000157540", "ENSG00000157542", "ENSG00000157557", "ENSG00000157600", "ENSG00000157601", "ENSG00000157613", "ENSG00000157617", "ENSG00000157625", "ENSG00000157637", "ENSG00000157657", "ENSG00000157741", "ENSG00000157764", "ENSG00000157778", "ENSG00000157796", "ENSG00000157800", "ENSG00000157827", "ENSG00000157837", "ENSG00000157851", "ENSG00000157869", "ENSG00000157895", "ENSG00000157916", "ENSG00000157954", "ENSG00000157985", "ENSG00000158062", "ENSG00000158092", "ENSG00000158109", "ENSG00000158161", "ENSG00000158186", "ENSG00000158195", "ENSG00000158201", "ENSG00000158258", "ENSG00000158290", "ENSG00000158321", "ENSG00000158417", "ENSG00000158435", "ENSG00000158445", "ENSG00000158457", "ENSG00000158467", "ENSG00000158470", "ENSG00000158480", "ENSG00000158526", "ENSG00000158528", "ENSG00000158555", "ENSG00000158560", "ENSG00000158604", "ENSG00000158615", "ENSG00000158623", "ENSG00000158716", "ENSG00000158793", "ENSG00000158796", "ENSG00000158828", "ENSG00000158850", "ENSG00000158856", "ENSG00000158864", "ENSG00000158882", "ENSG00000158941", "ENSG00000158985", "ENSG00000158987", "ENSG00000159063", "ENSG00000159082", "ENSG00000159086", "ENSG00000159111", "ENSG00000159128", "ENSG00000159140", "ENSG00000159164", "ENSG00000159167", "ENSG00000159176", "ENSG00000159199", "ENSG00000159200", "ENSG00000159202", "ENSG00000159210", "ENSG00000159228", "ENSG00000159256", "ENSG00000159307", "ENSG00000159322", "ENSG00000159335", "ENSG00000159348", "ENSG00000159352", "ENSG00000159363", "ENSG00000159377", "ENSG00000159409", "ENSG00000159423", "ENSG00000159445", "ENSG00000159459", "ENSG00000159461", "ENSG00000159479", "ENSG00000159579", "ENSG00000159592", "ENSG00000159593", "ENSG00000159658", "ENSG00000159685", "ENSG00000159692", "ENSG00000159720", "ENSG00000159784", "ENSG00000159788", "ENSG00000159840", "ENSG00000159842", "ENSG00000159921", "ENSG00000160007", "ENSG00000160014", "ENSG00000160049", "ENSG00000160058", "ENSG00000160075", "ENSG00000160087", "ENSG00000160113", "ENSG00000160131", "ENSG00000160145", "ENSG00000160179", "ENSG00000160191", "ENSG00000160194", "ENSG00000160209", "ENSG00000160213", "ENSG00000160214", "ENSG00000160216", "ENSG00000160285", "ENSG00000160294", "ENSG00000160299", "ENSG00000160305", "ENSG00000160307", "ENSG00000160310", "ENSG00000160321", "ENSG00000160326", "ENSG00000160439", "ENSG00000160445", "ENSG00000160460", "ENSG00000160469", "ENSG00000160551", "ENSG00000160570", "ENSG00000160584", "ENSG00000160613", "ENSG00000160633", "ENSG00000160679", "ENSG00000160688", "ENSG00000160695", "ENSG00000160710", "ENSG00000160714", "ENSG00000160716", "ENSG00000160746", "ENSG00000160753", "ENSG00000160785", "ENSG00000160789", "ENSG00000160799", "ENSG00000160818", "ENSG00000160917", "ENSG00000160948", "ENSG00000160959", "ENSG00000160961", "ENSG00000160991", "ENSG00000161011", "ENSG00000161021", "ENSG00000161048", "ENSG00000161057", "ENSG00000161082", "ENSG00000161202", "ENSG00000161203", "ENSG00000161204", "ENSG00000161217", "ENSG00000161267", "ENSG00000161526", "ENSG00000161533", "ENSG00000161542", "ENSG00000161642", "ENSG00000161813", "ENSG00000161980", "ENSG00000161981", "ENSG00000162065", "ENSG00000162076", "ENSG00000162104", "ENSG00000162105", "ENSG00000162174", "ENSG00000162188", "ENSG00000162191", "ENSG00000162222", "ENSG00000162298", "ENSG00000162368", "ENSG00000162374", "ENSG00000162378", "ENSG00000162402", "ENSG00000162408", "ENSG00000162409", "ENSG00000162413", "ENSG00000162433", "ENSG00000162434", "ENSG00000162437", "ENSG00000162511", "ENSG00000162512", "ENSG00000162517", "ENSG00000162521", "ENSG00000162545", "ENSG00000162585", "ENSG00000162599", "ENSG00000162601", "ENSG00000162604", "ENSG00000162607", "ENSG00000162613", "ENSG00000162616", "ENSG00000162623", "ENSG00000162630", "ENSG00000162642", "ENSG00000162664", "ENSG00000162688", "ENSG00000162694", "ENSG00000162695", "ENSG00000162702", "ENSG00000162704", "ENSG00000162706", "ENSG00000162728", "ENSG00000162729", "ENSG00000162734", "ENSG00000162735", "ENSG00000162736", "ENSG00000162769", "ENSG00000162819", "ENSG00000162849", "ENSG00000162851", "ENSG00000162852", "ENSG00000162869", "ENSG00000162873", "ENSG00000162889", "ENSG00000162909", "ENSG00000162910", "ENSG00000162923", "ENSG00000162961", "ENSG00000162980", "ENSG00000162989", "ENSG00000163013", "ENSG00000163029", "ENSG00000163032", "ENSG00000163053", "ENSG00000163064", "ENSG00000163069", "ENSG00000163104", "ENSG00000163110", "ENSG00000163125", "ENSG00000163156", "ENSG00000163159", "ENSG00000163166", "ENSG00000163170", "ENSG00000163214", "ENSG00000163257", "ENSG00000163288", "ENSG00000163291", "ENSG00000163320", "ENSG00000163328", "ENSG00000163344", "ENSG00000163346", "ENSG00000163349", "ENSG00000163374", "ENSG00000163380", "ENSG00000163389", "ENSG00000163393", "ENSG00000163399", "ENSG00000163412", "ENSG00000163444", "ENSG00000163453", "ENSG00000163466", "ENSG00000163468", "ENSG00000163479", "ENSG00000163513", "ENSG00000163517", "ENSG00000163528", "ENSG00000163531", "ENSG00000163536", "ENSG00000163539", "ENSG00000163541", "ENSG00000163558", "ENSG00000163577", "ENSG00000163590", "ENSG00000163596", "ENSG00000163602", "ENSG00000163618", "ENSG00000163624", "ENSG00000163625", "ENSG00000163629", "ENSG00000163630", "ENSG00000163634", "ENSG00000163635", "ENSG00000163636", "ENSG00000163637", "ENSG00000163644", "ENSG00000163655", "ENSG00000163660", "ENSG00000163681", "ENSG00000163683", "ENSG00000163697", "ENSG00000163704", "ENSG00000163714", "ENSG00000163719", "ENSG00000163728", "ENSG00000163743", "ENSG00000163754", "ENSG00000163781", "ENSG00000163788", "ENSG00000163807", "ENSG00000163811", "ENSG00000163812", "ENSG00000163818", "ENSG00000163832", "ENSG00000163848", "ENSG00000163866", "ENSG00000163867", "ENSG00000163872", "ENSG00000163873", "ENSG00000163875", "ENSG00000163882", "ENSG00000163900", "ENSG00000163902", "ENSG00000163904", "ENSG00000163931", "ENSG00000163939", "ENSG00000163947", "ENSG00000163956", "ENSG00000163960", "ENSG00000163961", "ENSG00000163964", "ENSG00000163995", "ENSG00000164022", "ENSG00000164031", "ENSG00000164038", "ENSG00000164040", "ENSG00000164050", "ENSG00000164051", "ENSG00000164054", "ENSG00000164061", "ENSG00000164062", "ENSG00000164066", "ENSG00000164070", "ENSG00000164076", "ENSG00000164080", "ENSG00000164089", "ENSG00000164091", "ENSG00000164096", "ENSG00000164104", "ENSG00000164106", "ENSG00000164111", "ENSG00000164114", "ENSG00000164117", "ENSG00000164124", "ENSG00000164134", "ENSG00000164151", "ENSG00000164163", "ENSG00000164164", "ENSG00000164167", "ENSG00000164172", "ENSG00000164176", "ENSG00000164181", "ENSG00000164187", "ENSG00000164190", "ENSG00000164197", "ENSG00000164199", "ENSG00000164209", "ENSG00000164211", "ENSG00000164252", "ENSG00000164253", "ENSG00000164258", "ENSG00000164284", "ENSG00000164292", "ENSG00000164323", "ENSG00000164327", "ENSG00000164330", "ENSG00000164331", "ENSG00000164332", "ENSG00000164346", "ENSG00000164366", "ENSG00000164398", "ENSG00000164405", "ENSG00000164418", "ENSG00000164434", "ENSG00000164463", "ENSG00000164466", "ENSG00000164506", "ENSG00000164548", "ENSG00000164574", "ENSG00000164576", "ENSG00000164587", "ENSG00000164588", "ENSG00000164604", "ENSG00000164609", "ENSG00000164615", "ENSG00000164631", "ENSG00000164638", "ENSG00000164654", "ENSG00000164683", "ENSG00000164733", "ENSG00000164741", "ENSG00000164742", "ENSG00000164751", "ENSG00000164754", "ENSG00000164778", "ENSG00000164796", "ENSG00000164808", "ENSG00000164815", "ENSG00000164823", "ENSG00000164828", "ENSG00000164830", "ENSG00000164896", "ENSG00000164897", "ENSG00000164902", "ENSG00000164904", "ENSG00000164916", "ENSG00000164919", "ENSG00000164924", "ENSG00000164929", "ENSG00000164934", "ENSG00000164951", "ENSG00000164970", "ENSG00000164975", "ENSG00000164978", "ENSG00000164985", "ENSG00000165006", "ENSG00000165023", "ENSG00000165028", "ENSG00000165029", "ENSG00000165092", "ENSG00000165102", "ENSG00000165113", "ENSG00000165138", "ENSG00000165156", "ENSG00000165169", "ENSG00000165175", "ENSG00000165185", "ENSG00000165186", "ENSG00000165209", "ENSG00000165219", "ENSG00000165238", "ENSG00000165240", "ENSG00000165264", "ENSG00000165271", "ENSG00000165280", "ENSG00000165300", "ENSG00000165322", "ENSG00000165323", "ENSG00000165338", "ENSG00000165355", "ENSG00000165379", "ENSG00000165410", "ENSG00000165416", "ENSG00000165417", "ENSG00000165434", "ENSG00000165443", "ENSG00000165475", "ENSG00000165476", "ENSG00000165487", "ENSG00000165495", "ENSG00000165516", "ENSG00000165525", "ENSG00000165566", "ENSG00000165572", "ENSG00000165609", "ENSG00000165632", "ENSG00000165644", "ENSG00000165646", "ENSG00000165650", "ENSG00000165655", "ENSG00000165669", "ENSG00000165671", "ENSG00000165672", "ENSG00000165678", "ENSG00000165699", "ENSG00000165704", "ENSG00000165724", "ENSG00000165731", "ENSG00000165732", "ENSG00000165733", "ENSG00000165775", "ENSG00000165795", "ENSG00000165801", "ENSG00000165802", "ENSG00000165813", "ENSG00000165821", "ENSG00000165832", "ENSG00000165861", "ENSG00000165868", "ENSG00000165898", "ENSG00000165914", "ENSG00000165915", "ENSG00000165916", "ENSG00000165934", "ENSG00000165943", "ENSG00000165995", "ENSG00000165997", "ENSG00000166002", "ENSG00000166012", "ENSG00000166033", "ENSG00000166037", "ENSG00000166073", "ENSG00000166111", "ENSG00000166128", "ENSG00000166135", "ENSG00000166166", "ENSG00000166167", "ENSG00000166170", "ENSG00000166173", "ENSG00000166181", "ENSG00000166200", "ENSG00000166206", "ENSG00000166224", "ENSG00000166226", "ENSG00000166228", "ENSG00000166233", "ENSG00000166257", "ENSG00000166260", "ENSG00000166266", "ENSG00000166272", "ENSG00000166295", "ENSG00000166311", "ENSG00000166326", "ENSG00000166337", "ENSG00000166340", "ENSG00000166347", "ENSG00000166348", "ENSG00000166402", "ENSG00000166405", "ENSG00000166432", "ENSG00000166446", "ENSG00000166448", "ENSG00000166452", "ENSG00000166454", "ENSG00000166471", "ENSG00000166479", "ENSG00000166501", "ENSG00000166532", "ENSG00000166548", "ENSG00000166557", "ENSG00000166562", "ENSG00000166579", "ENSG00000166598", "ENSG00000166619", "ENSG00000166676", "ENSG00000166710", "ENSG00000166747", "ENSG00000166770", "ENSG00000166822", "ENSG00000166839", "ENSG00000166847", "ENSG00000166848", "ENSG00000166855", "ENSG00000166887", "ENSG00000166900", "ENSG00000166902", "ENSG00000166908", "ENSG00000166913", "ENSG00000166925", "ENSG00000166946", "ENSG00000166963", "ENSG00000166971", "ENSG00000166974", "ENSG00000167004", "ENSG00000167037", "ENSG00000167074", "ENSG00000167081", "ENSG00000167088", "ENSG00000167110", "ENSG00000167112", "ENSG00000167113", "ENSG00000167118", "ENSG00000167191", "ENSG00000167193", "ENSG00000167196", "ENSG00000167232", "ENSG00000167258", "ENSG00000167280", "ENSG00000167291", "ENSG00000167323", "ENSG00000167325", "ENSG00000167371", "ENSG00000167378", "ENSG00000167380", "ENSG00000167460", "ENSG00000167468", "ENSG00000167470", "ENSG00000167491", "ENSG00000167515", "ENSG00000167522", "ENSG00000167526", "ENSG00000167535", "ENSG00000167548", "ENSG00000167549", "ENSG00000167552", "ENSG00000167555", "ENSG00000167601", "ENSG00000167614", "ENSG00000167632", "ENSG00000167635", "ENSG00000167642", "ENSG00000167645", "ENSG00000167654", "ENSG00000167657", "ENSG00000167658", "ENSG00000167671", "ENSG00000167680", "ENSG00000167685", "ENSG00000167699", "ENSG00000167700", "ENSG00000167733", "ENSG00000167770", "ENSG00000167778", "ENSG00000167779", "ENSG00000167792", "ENSG00000167815", "ENSG00000167861", "ENSG00000167881", "ENSG00000167964", "ENSG00000167971", "ENSG00000167972", "ENSG00000167978", "ENSG00000167986", "ENSG00000168002", "ENSG00000168003", "ENSG00000168036", "ENSG00000168066", "ENSG00000168090", "ENSG00000168092", "ENSG00000168118", "ENSG00000168137", "ENSG00000168159", "ENSG00000168172", "ENSG00000168175", "ENSG00000168214", "ENSG00000168216", "ENSG00000168234", "ENSG00000168243", "ENSG00000168264", "ENSG00000168275", "ENSG00000168280", "ENSG00000168286", "ENSG00000168288", "ENSG00000168291", "ENSG00000168297", "ENSG00000168301", "ENSG00000168303", "ENSG00000168309", "ENSG00000168314", "ENSG00000168374", "ENSG00000168395", "ENSG00000168397", "ENSG00000168434", "ENSG00000168453", "ENSG00000168461", "ENSG00000168481", "ENSG00000168495", "ENSG00000168502", "ENSG00000168522", "ENSG00000168566", "ENSG00000168591", "ENSG00000168610", "ENSG00000168615", "ENSG00000168653", "ENSG00000168675", "ENSG00000168701", "ENSG00000168702", "ENSG00000168710", "ENSG00000168724", "ENSG00000168734", "ENSG00000168743", "ENSG00000168772", "ENSG00000168781", "ENSG00000168785", "ENSG00000168818", "ENSG00000168824", "ENSG00000168827", "ENSG00000168843", "ENSG00000168883", "ENSG00000168904", "ENSG00000168906", "ENSG00000168913", "ENSG00000168917", "ENSG00000168924", "ENSG00000168936", "ENSG00000168958", "ENSG00000168993", "ENSG00000169006", "ENSG00000169018", "ENSG00000169019", "ENSG00000169032", "ENSG00000169045", "ENSG00000169047", "ENSG00000169057", "ENSG00000169083", "ENSG00000169116", "ENSG00000169122", "ENSG00000169139", "ENSG00000169155", "ENSG00000169169", "ENSG00000169180", "ENSG00000169184", "ENSG00000169189", "ENSG00000169193", "ENSG00000169213", "ENSG00000169217", "ENSG00000169223", "ENSG00000169251", "ENSG00000169255", "ENSG00000169288", "ENSG00000169302", "ENSG00000169359", "ENSG00000169375", "ENSG00000169398", "ENSG00000169410", "ENSG00000169432", "ENSG00000169439", "ENSG00000169446", "ENSG00000169490", "ENSG00000169504", "ENSG00000169554", "ENSG00000169564", "ENSG00000169567", "ENSG00000169641", "ENSG00000169710", "ENSG00000169714", "ENSG00000169740", "ENSG00000169744", "ENSG00000169760", "ENSG00000169762", "ENSG00000169764", "ENSG00000169783", "ENSG00000169826", "ENSG00000169851", "ENSG00000169855", "ENSG00000169862", "ENSG00000169891", "ENSG00000169902", "ENSG00000169905", "ENSG00000169926", "ENSG00000169933", "ENSG00000169967", "ENSG00000169976", "ENSG00000169992", "ENSG00000170011", "ENSG00000170017", "ENSG00000170027", "ENSG00000170075", "ENSG00000170088", "ENSG00000170113", "ENSG00000170142", "ENSG00000170144", "ENSG00000170145", "ENSG00000170153", "ENSG00000170231", "ENSG00000170234", "ENSG00000170242", "ENSG00000170248", "ENSG00000170266", "ENSG00000170275", "ENSG00000170310", "ENSG00000170315", "ENSG00000170340", "ENSG00000170348", "ENSG00000170385", "ENSG00000170419", "ENSG00000170448", "ENSG00000170456", "ENSG00000170464", "ENSG00000170471", "ENSG00000170500", "ENSG00000170502", "ENSG00000170515", "ENSG00000170522", "ENSG00000170525", "ENSG00000170540", "ENSG00000170542", "ENSG00000170558", "ENSG00000170571", "ENSG00000170581", "ENSG00000170606", "ENSG00000170619", "ENSG00000170624", "ENSG00000170633", "ENSG00000170743", "ENSG00000170745", "ENSG00000170759", "ENSG00000170776", "ENSG00000170832", "ENSG00000170852", "ENSG00000170860", "ENSG00000170871", "ENSG00000170873", "ENSG00000170876", "ENSG00000170881", "ENSG00000170889", "ENSG00000170899", "ENSG00000170903", "ENSG00000170906", "ENSG00000170915", "ENSG00000170921", "ENSG00000170954", "ENSG00000170989", "ENSG00000171004", "ENSG00000171033", "ENSG00000171055", "ENSG00000171067", "ENSG00000171105", "ENSG00000171109", "ENSG00000171130", "ENSG00000171135", "ENSG00000171150", "ENSG00000171155", "ENSG00000171163", "ENSG00000171202", "ENSG00000171204", "ENSG00000171206", "ENSG00000171208", "ENSG00000171222", "ENSG00000171223", "ENSG00000171262", "ENSG00000171298", "ENSG00000171302", "ENSG00000171307", "ENSG00000171310", "ENSG00000171311", "ENSG00000171365", "ENSG00000171368", "ENSG00000171385", "ENSG00000171435", "ENSG00000171444", "ENSG00000171450", "ENSG00000171451", "ENSG00000171456", "ENSG00000171467", "ENSG00000171475", "ENSG00000171490", "ENSG00000171492", "ENSG00000171503", "ENSG00000171533", "ENSG00000171552", "ENSG00000171566", "ENSG00000171587", "ENSG00000171603", "ENSG00000171604", "ENSG00000171608", "ENSG00000171617", "ENSG00000171634", "ENSG00000171681", "ENSG00000171703", "ENSG00000171720", "ENSG00000171723", "ENSG00000171724", "ENSG00000171735", "ENSG00000171766", "ENSG00000171793", "ENSG00000171824", "ENSG00000171843", "ENSG00000171858", "ENSG00000171862", "ENSG00000171867", "ENSG00000171885", "ENSG00000171914", "ENSG00000171951", "ENSG00000171953", "ENSG00000171988", "ENSG00000172007", "ENSG00000172020", "ENSG00000172046", "ENSG00000172057", "ENSG00000172137", "ENSG00000172201", "ENSG00000172260", "ENSG00000172262", "ENSG00000172264", "ENSG00000172269", "ENSG00000172292", "ENSG00000172301", "ENSG00000172331", "ENSG00000172336", "ENSG00000172339", "ENSG00000172346", "ENSG00000172348", "ENSG00000172350", "ENSG00000172379", "ENSG00000172403", "ENSG00000172461", "ENSG00000172466", "ENSG00000172493", "ENSG00000172500", "ENSG00000172508", "ENSG00000172534", "ENSG00000172586", "ENSG00000172667", "ENSG00000172731", "ENSG00000172757", "ENSG00000172765", "ENSG00000172794", "ENSG00000172795", "ENSG00000172809", "ENSG00000172831", "ENSG00000172845", "ENSG00000172869", "ENSG00000172889", "ENSG00000172893", "ENSG00000172915", "ENSG00000172922", "ENSG00000172932", "ENSG00000172939", "ENSG00000172992", "ENSG00000173011", "ENSG00000173020", "ENSG00000173039", "ENSG00000173064", "ENSG00000173065", "ENSG00000173068", "ENSG00000173114", "ENSG00000173120", "ENSG00000173141", "ENSG00000173210", "ENSG00000173221", "ENSG00000173230", "ENSG00000173267", "ENSG00000173272", "ENSG00000173273", "ENSG00000173276", "ENSG00000173320", "ENSG00000173402", "ENSG00000173404", "ENSG00000173406", "ENSG00000173409", "ENSG00000173418", "ENSG00000173482", "ENSG00000173517", "ENSG00000173545", "ENSG00000173575", "ENSG00000173611", "ENSG00000173674", "ENSG00000173692", "ENSG00000173726", "ENSG00000173744", "ENSG00000173757", "ENSG00000173786", "ENSG00000173812", "ENSG00000173821", "ENSG00000173852", "ENSG00000173875", "ENSG00000173889", "ENSG00000173898", "ENSG00000173905", "ENSG00000173914", "ENSG00000173992", "ENSG00000174013", "ENSG00000174080", "ENSG00000174106", "ENSG00000174132", "ENSG00000174136", "ENSG00000174165", "ENSG00000174173", "ENSG00000174197", "ENSG00000174238", "ENSG00000174282", "ENSG00000174306", "ENSG00000174405", "ENSG00000174437", "ENSG00000174446", "ENSG00000174456", "ENSG00000174460", "ENSG00000174469", "ENSG00000174516", "ENSG00000174521", "ENSG00000174574", "ENSG00000174579", "ENSG00000174606", "ENSG00000174607", "ENSG00000174628", "ENSG00000174684", "ENSG00000174695", "ENSG00000174720", "ENSG00000174738", "ENSG00000174748", "ENSG00000174775", "ENSG00000174780", "ENSG00000174840", "ENSG00000174886", "ENSG00000174891", "ENSG00000174915", "ENSG00000174938", "ENSG00000174939", "ENSG00000174943", "ENSG00000174953", "ENSG00000175029", "ENSG00000175048", "ENSG00000175054", "ENSG00000175073", "ENSG00000175105", "ENSG00000175110", "ENSG00000175115", "ENSG00000175130", "ENSG00000175161", "ENSG00000175166", "ENSG00000175198", "ENSG00000175203", "ENSG00000175215", "ENSG00000175216", "ENSG00000175220", "ENSG00000175221", "ENSG00000175224", "ENSG00000175264", "ENSG00000175265", "ENSG00000175283", "ENSG00000175334", "ENSG00000175348", "ENSG00000175352", "ENSG00000175376", "ENSG00000175387", "ENSG00000175395", "ENSG00000175416", "ENSG00000175426", "ENSG00000175470", "ENSG00000175497", "ENSG00000175573", "ENSG00000175575", "ENSG00000175581", "ENSG00000175602", "ENSG00000175606", "ENSG00000175662", "ENSG00000175727", "ENSG00000175756", "ENSG00000175782", "ENSG00000175785", "ENSG00000175806", "ENSG00000175826", "ENSG00000175866", "ENSG00000175874", "ENSG00000175893", "ENSG00000175931", "ENSG00000176022", "ENSG00000176049", "ENSG00000176055", "ENSG00000176101", "ENSG00000176102", "ENSG00000176148", "ENSG00000176171", "ENSG00000176222", "ENSG00000176340", "ENSG00000176390", "ENSG00000176396", "ENSG00000176401", "ENSG00000176406", "ENSG00000176407", "ENSG00000176410", "ENSG00000176454", "ENSG00000176463", "ENSG00000176533", "ENSG00000176595", "ENSG00000176623", "ENSG00000176624", "ENSG00000176697", "ENSG00000176720", "ENSG00000176749", "ENSG00000176783", "ENSG00000176788", "ENSG00000176871", "ENSG00000176884", "ENSG00000176887", "ENSG00000176946", "ENSG00000176953", "ENSG00000176956", "ENSG00000176978", "ENSG00000176986", "ENSG00000177000", "ENSG00000177054", "ENSG00000177082", "ENSG00000177098", "ENSG00000177108", "ENSG00000177150", "ENSG00000177156", "ENSG00000177181", "ENSG00000177189", "ENSG00000177200", "ENSG00000177239", "ENSG00000177301", "ENSG00000177311", "ENSG00000177370", "ENSG00000177383", "ENSG00000177410", "ENSG00000177432", "ENSG00000177479", "ENSG00000177542", "ENSG00000177551", "ENSG00000177556", "ENSG00000177565", "ENSG00000177600", "ENSG00000177606", "ENSG00000177613", "ENSG00000177614", "ENSG00000177646", "ENSG00000177683", "ENSG00000177697", "ENSG00000177700", "ENSG00000177706", "ENSG00000177731", "ENSG00000177733", "ENSG00000177830", "ENSG00000177875", "ENSG00000177885", "ENSG00000177888", "ENSG00000177889", "ENSG00000177951", "ENSG00000177963", "ENSG00000177971", "ENSG00000177981", "ENSG00000178053", "ENSG00000178057", "ENSG00000178074", "ENSG00000178104", "ENSG00000178163", "ENSG00000178171", "ENSG00000178234", "ENSG00000178235", "ENSG00000178301", "ENSG00000178307", "ENSG00000178381", "ENSG00000178425", "ENSG00000178449", "ENSG00000178498", "ENSG00000178538", "ENSG00000178567", "ENSG00000178568", "ENSG00000178662", "ENSG00000178691", "ENSG00000178695", "ENSG00000178718", "ENSG00000178741", "ENSG00000178761", "ENSG00000178913", "ENSG00000178947", "ENSG00000178950", "ENSG00000178951", "ENSG00000178952", "ENSG00000178965", "ENSG00000178974", "ENSG00000178982", "ENSG00000178988", "ENSG00000179010", "ENSG00000179021", "ENSG00000179083", "ENSG00000179085", "ENSG00000179091", "ENSG00000179115", "ENSG00000179134", "ENSG00000179195", "ENSG00000179218", "ENSG00000179222", "ENSG00000179242", "ENSG00000179262", "ENSG00000179295", "ENSG00000179314", "ENSG00000179364", "ENSG00000179387", "ENSG00000179431", "ENSG00000179454", "ENSG00000179456", "ENSG00000179526", "ENSG00000179542", "ENSG00000179598", "ENSG00000179796", "ENSG00000179820", "ENSG00000179889", "ENSG00000179912", "ENSG00000179915", "ENSG00000179918", "ENSG00000179933", "ENSG00000179950", "ENSG00000179958", "ENSG00000179981", "ENSG00000180008", "ENSG00000180104", "ENSG00000180155", "ENSG00000180176", "ENSG00000180182", "ENSG00000180185", "ENSG00000180190", "ENSG00000180228", "ENSG00000180304", "ENSG00000180354", "ENSG00000180370", "ENSG00000180398", "ENSG00000180543", "ENSG00000180694", "ENSG00000180817", "ENSG00000180822", "ENSG00000180879", "ENSG00000180881", "ENSG00000180901", "ENSG00000180902", "ENSG00000180964", "ENSG00000181019", "ENSG00000181035", "ENSG00000181045", "ENSG00000181061", "ENSG00000181090", "ENSG00000181163", "ENSG00000181191", "ENSG00000181192", "ENSG00000181222", "ENSG00000181234", "ENSG00000181409", "ENSG00000181449", "ENSG00000181523", "ENSG00000181555", "ENSG00000181610", "ENSG00000181638", "ENSG00000181704", "ENSG00000181754", "ENSG00000181789", "ENSG00000181790", "ENSG00000181827", "ENSG00000181830", "ENSG00000181852", "ENSG00000181904", "ENSG00000181924", "ENSG00000181991", "ENSG00000182054", "ENSG00000182087", "ENSG00000182117", "ENSG00000182134", "ENSG00000182149", "ENSG00000182154", "ENSG00000182168", "ENSG00000182195", "ENSG00000182199", "ENSG00000182208", "ENSG00000182220", "ENSG00000182247", "ENSG00000182253", "ENSG00000182287", "ENSG00000182307", "ENSG00000182372", "ENSG00000182400", "ENSG00000182446", "ENSG00000182504", "ENSG00000182512", "ENSG00000182534", "ENSG00000182541", "ENSG00000182551", "ENSG00000182568", "ENSG00000182606", "ENSG00000182621", "ENSG00000182628", "ENSG00000182636", "ENSG00000182667", "ENSG00000182670", "ENSG00000182718", "ENSG00000182732", "ENSG00000182747", "ENSG00000182752", "ENSG00000182771", "ENSG00000182827", "ENSG00000182836", "ENSG00000182870", "ENSG00000182872", "ENSG00000182899", "ENSG00000182901", "ENSG00000182902", "ENSG00000182903", "ENSG00000182916", "ENSG00000182919", "ENSG00000182923", "ENSG00000182944", "ENSG00000182985", "ENSG00000183011", "ENSG00000183020", "ENSG00000183023", "ENSG00000183036", "ENSG00000183044", "ENSG00000183049", "ENSG00000183087", "ENSG00000183091", "ENSG00000183092", "ENSG00000183111", "ENSG00000183114", "ENSG00000183117", "ENSG00000183155", "ENSG00000183166", "ENSG00000183172", "ENSG00000183207", "ENSG00000183255", "ENSG00000183258", "ENSG00000183283", "ENSG00000183317", "ENSG00000183431", "ENSG00000183454", "ENSG00000183495", "ENSG00000183513", "ENSG00000183530", "ENSG00000183570", "ENSG00000183576", "ENSG00000183597", "ENSG00000183624", "ENSG00000183648", "ENSG00000183723", "ENSG00000183726", "ENSG00000183735", "ENSG00000183741", "ENSG00000183751", "ENSG00000183779", "ENSG00000183826", "ENSG00000183864", "ENSG00000183978", "ENSG00000184007", "ENSG00000184009", "ENSG00000184014", "ENSG00000184076", "ENSG00000184117", "ENSG00000184144", "ENSG00000184156", "ENSG00000184164", "ENSG00000184178", "ENSG00000184182", "ENSG00000184194", "ENSG00000184203", "ENSG00000184205", "ENSG00000184209", "ENSG00000184216", "ENSG00000184220", "ENSG00000184221", "ENSG00000184226", "ENSG00000184271", "ENSG00000184277", "ENSG00000184349", "ENSG00000184368", "ENSG00000184388", "ENSG00000184402", "ENSG00000184408", "ENSG00000184432", "ENSG00000184436", "ENSG00000184486", "ENSG00000184515", "ENSG00000184517", "ENSG00000184524", "ENSG00000184545", "ENSG00000184575", "ENSG00000184588", "ENSG00000184602", "ENSG00000184613", "ENSG00000184640", "ENSG00000184672", "ENSG00000184677", "ENSG00000184708", "ENSG00000184752", "ENSG00000184792", "ENSG00000184831", "ENSG00000184840", "ENSG00000184863", "ENSG00000184867", "ENSG00000184887", "ENSG00000184900", "ENSG00000184905", "ENSG00000184916", "ENSG00000184939", "ENSG00000184983", "ENSG00000184992", "ENSG00000185008", "ENSG00000185009", "ENSG00000185010", "ENSG00000185024", "ENSG00000185043", "ENSG00000185046", "ENSG00000185049", "ENSG00000185052", "ENSG00000185070", "ENSG00000185088", "ENSG00000185090", "ENSG00000185104", "ENSG00000185127", "ENSG00000185129", "ENSG00000185187", "ENSG00000185219", "ENSG00000185246", "ENSG00000185262", "ENSG00000185324", "ENSG00000185340", "ENSG00000185359", "ENSG00000185420", "ENSG00000185442", "ENSG00000185477", "ENSG00000185513", "ENSG00000185518", "ENSG00000185559", "ENSG00000185565", "ENSG00000185567", "ENSG00000185608", "ENSG00000185619", "ENSG00000185624", "ENSG00000185627", "ENSG00000185630", "ENSG00000185650", "ENSG00000185651", "ENSG00000185658", "ENSG00000185722", "ENSG00000185728", "ENSG00000185742", "ENSG00000185745", "ENSG00000185774", "ENSG00000185787", "ENSG00000185800", "ENSG00000185803", "ENSG00000185808", "ENSG00000185813", "ENSG00000185818", "ENSG00000185825", "ENSG00000185880", "ENSG00000185896", "ENSG00000185904", "ENSG00000185909", "ENSG00000185917", "ENSG00000185920", "ENSG00000185946", "ENSG00000185950", "ENSG00000185963", "ENSG00000186001", "ENSG00000186020", "ENSG00000186106", "ENSG00000186111", "ENSG00000186184", "ENSG00000186187", "ENSG00000186298", "ENSG00000186310", "ENSG00000186318", "ENSG00000186340", "ENSG00000186350", "ENSG00000186377", "ENSG00000186395", "ENSG00000186416", "ENSG00000186432", "ENSG00000186462", "ENSG00000186468", "ENSG00000186469", "ENSG00000186472", "ENSG00000186480", "ENSG00000186487", "ENSG00000186501", "ENSG00000186566", "ENSG00000186575", "ENSG00000186591", "ENSG00000186642", "ENSG00000186660", "ENSG00000186665", "ENSG00000186687", "ENSG00000186732", "ENSG00000186806", "ENSG00000186815", "ENSG00000186834", "ENSG00000186866", "ENSG00000186868", "ENSG00000186908", "ENSG00000187068", "ENSG00000187094", "ENSG00000187098", "ENSG00000187109", "ENSG00000187147", "ENSG00000187164", "ENSG00000187189", "ENSG00000187193", "ENSG00000187231", "ENSG00000187239", "ENSG00000187240", "ENSG00000187257", "ENSG00000187391", "ENSG00000187398", "ENSG00000187446", "ENSG00000187522", "ENSG00000187555", "ENSG00000187601", "ENSG00000187672", "ENSG00000187713", "ENSG00000187735", "ENSG00000187742", "ENSG00000187764", "ENSG00000187957", "ENSG00000188021", "ENSG00000188042", "ENSG00000188130", "ENSG00000188186", "ENSG00000188191", "ENSG00000188227", "ENSG00000188229", "ENSG00000188243", "ENSG00000188342", "ENSG00000188385", "ENSG00000188419", "ENSG00000188554", "ENSG00000188559", "ENSG00000188580", "ENSG00000188612", "ENSG00000188647", "ENSG00000188674", "ENSG00000188677", "ENSG00000188690", "ENSG00000188706", "ENSG00000188725", "ENSG00000188803", "ENSG00000188811", "ENSG00000188846", "ENSG00000188848", "ENSG00000188895", "ENSG00000188938", "ENSG00000188986", "ENSG00000188994", "ENSG00000188997", "ENSG00000189058", "ENSG00000189067", "ENSG00000189079", "ENSG00000189091", "ENSG00000189171", "ENSG00000189180", "ENSG00000189184", "ENSG00000189221", "ENSG00000189227", "ENSG00000189241", "ENSG00000189337", "ENSG00000189369", "ENSG00000196072", "ENSG00000196091", "ENSG00000196104", "ENSG00000196116", "ENSG00000196132", "ENSG00000196141", "ENSG00000196177", "ENSG00000196199", "ENSG00000196220", "ENSG00000196227", "ENSG00000196233", "ENSG00000196235", "ENSG00000196262", "ENSG00000196290", "ENSG00000196323", "ENSG00000196338", "ENSG00000196352", "ENSG00000196361", "ENSG00000196363", "ENSG00000196367", "ENSG00000196368", "ENSG00000196372", "ENSG00000196376", "ENSG00000196405", "ENSG00000196422", "ENSG00000196428", "ENSG00000196440", "ENSG00000196449", "ENSG00000196458", "ENSG00000196459", "ENSG00000196465", "ENSG00000196470", "ENSG00000196482", "ENSG00000196498", "ENSG00000196504", "ENSG00000196507", "ENSG00000196510", "ENSG00000196531", "ENSG00000196547", "ENSG00000196562", "ENSG00000196581", "ENSG00000196586", "ENSG00000196591", "ENSG00000196628", "ENSG00000196632", "ENSG00000196642", "ENSG00000196655", "ENSG00000196663", "ENSG00000196683", "ENSG00000196704", "ENSG00000196712", "ENSG00000196715", "ENSG00000196730", "ENSG00000196776", "ENSG00000196781", "ENSG00000196792", "ENSG00000196814", "ENSG00000196850", "ENSG00000196876", "ENSG00000196911", "ENSG00000196914", "ENSG00000196923", "ENSG00000196937", "ENSG00000196950", "ENSG00000196961", "ENSG00000196967", "ENSG00000196968", "ENSG00000196998", "ENSG00000197006", "ENSG00000197043", "ENSG00000197045", "ENSG00000197063", "ENSG00000197077", "ENSG00000197102", "ENSG00000197111", "ENSG00000197121", "ENSG00000197147", "ENSG00000197150", "ENSG00000197157", "ENSG00000197170", "ENSG00000197177", "ENSG00000197183", "ENSG00000197217", "ENSG00000197226", "ENSG00000197265", "ENSG00000197296", "ENSG00000197323", "ENSG00000197345", "ENSG00000197380", "ENSG00000197381", "ENSG00000197442", "ENSG00000197444", "ENSG00000197448", "ENSG00000197457", "ENSG00000197461", "ENSG00000197498", "ENSG00000197535", "ENSG00000197548", "ENSG00000197555", "ENSG00000197580", "ENSG00000197586", "ENSG00000197594", "ENSG00000197601", "ENSG00000197622", "ENSG00000197694", "ENSG00000197702", "ENSG00000197746", "ENSG00000197756", "ENSG00000197771", "ENSG00000197780", "ENSG00000197858", "ENSG00000197860", "ENSG00000197885", "ENSG00000197894", "ENSG00000197912", "ENSG00000197930", "ENSG00000197956", "ENSG00000197959", "ENSG00000197961", "ENSG00000197965", "ENSG00000197969", "ENSG00000197971", "ENSG00000198000", "ENSG00000198015", "ENSG00000198034", "ENSG00000198042", "ENSG00000198046", "ENSG00000198053", "ENSG00000198055", "ENSG00000198105", "ENSG00000198121", "ENSG00000198130", "ENSG00000198142", "ENSG00000198160", "ENSG00000198162", "ENSG00000198168", "ENSG00000198171", "ENSG00000198198", "ENSG00000198216", "ENSG00000198231", "ENSG00000198258", "ENSG00000198265", "ENSG00000198300", "ENSG00000198301", "ENSG00000198315", "ENSG00000198363", "ENSG00000198373", "ENSG00000198380", "ENSG00000198382", "ENSG00000198399", "ENSG00000198417", "ENSG00000198420", "ENSG00000198431", "ENSG00000198478", "ENSG00000198483", "ENSG00000198513", "ENSG00000198522", "ENSG00000198585", "ENSG00000198586", "ENSG00000198612", "ENSG00000198624", "ENSG00000198625", "ENSG00000198642", "ENSG00000198646", "ENSG00000198648", "ENSG00000198663", "ENSG00000198689", "ENSG00000198700", "ENSG00000198721", "ENSG00000198722", "ENSG00000198730", "ENSG00000198740", "ENSG00000198742", "ENSG00000198780", "ENSG00000198785", "ENSG00000198791", "ENSG00000198792", "ENSG00000198794", "ENSG00000198799", "ENSG00000198815", "ENSG00000198818", "ENSG00000198825", "ENSG00000198833", "ENSG00000198836", "ENSG00000198837", "ENSG00000198838", "ENSG00000198853", "ENSG00000198856", "ENSG00000198860", "ENSG00000198862", "ENSG00000198863", "ENSG00000198876", "ENSG00000198894", "ENSG00000198898", "ENSG00000198900", "ENSG00000198910", "ENSG00000198911", "ENSG00000198912", "ENSG00000198915", "ENSG00000198919", "ENSG00000198929", "ENSG00000198931", "ENSG00000198932", "ENSG00000198934", "ENSG00000198937", "ENSG00000198947", "ENSG00000198948", "ENSG00000198961", "ENSG00000198964", "ENSG00000203485", "ENSG00000203705", "ENSG00000203778", "ENSG00000203875", "ENSG00000203877", "ENSG00000203879", "ENSG00000203880", "ENSG00000203930", "ENSG00000204070", "ENSG00000204104", "ENSG00000204120", "ENSG00000204128", "ENSG00000204130", "ENSG00000204175", "ENSG00000204186", "ENSG00000204217", "ENSG00000204237", "ENSG00000204370", "ENSG00000204406", "ENSG00000204842", "ENSG00000204843", "ENSG00000204856", "ENSG00000204899", "ENSG00000204922", "ENSG00000204977", "ENSG00000204991", "ENSG00000205060", "ENSG00000205138", "ENSG00000205250", "ENSG00000205269", "ENSG00000205279", "ENSG00000205302", "ENSG00000205336", "ENSG00000205339", "ENSG00000205356", "ENSG00000205364", "ENSG00000205423", "ENSG00000205531", "ENSG00000205629", "ENSG00000205659", "ENSG00000205726", "ENSG00000205758", "ENSG00000205937", "ENSG00000205981", "ENSG00000206052", "ENSG00000206418", "ENSG00000206432", "ENSG00000206538", "ENSG00000206560", "ENSG00000206573", "ENSG00000211445", "ENSG00000211455", "ENSG00000211456", "ENSG00000211460", "ENSG00000211584", "ENSG00000213015", "ENSG00000213024", "ENSG00000213047", "ENSG00000213064", "ENSG00000213079", "ENSG00000213190", "ENSG00000213281", "ENSG00000213341", "ENSG00000213465", "ENSG00000213585", "ENSG00000213593", "ENSG00000213614", "ENSG00000213619", "ENSG00000213625", "ENSG00000213639", "ENSG00000213672", "ENSG00000213699", "ENSG00000214022", "ENSG00000214026", "ENSG00000214078", "ENSG00000214113", "ENSG00000214413", "ENSG00000214517", "ENSG00000214548", "ENSG00000214753", "ENSG00000214941", "ENSG00000214944", "ENSG00000215021", "ENSG00000215114", "ENSG00000215193", "ENSG00000215218", "ENSG00000215301", "ENSG00000215305", "ENSG00000215475", "ENSG00000215712", "ENSG00000215717", "ENSG00000217128", "ENSG00000218739", "ENSG00000218891", "ENSG00000219481", "ENSG00000219545", "ENSG00000219626", "ENSG00000220205", "ENSG00000221823", "ENSG00000221866", "ENSG00000221914", "ENSG00000221978", "ENSG00000221983", "ENSG00000223802", "ENSG00000224032", "ENSG00000224051", "ENSG00000224470", "ENSG00000224531", "ENSG00000225465", "ENSG00000225733", "ENSG00000225783", "ENSG00000225968", "ENSG00000227051", "ENSG00000227345", "ENSG00000228474", "ENSG00000230989", "ENSG00000231721", "ENSG00000232119", "ENSG00000232859", "ENSG00000232956", "ENSG00000233016", "ENSG00000234456", "ENSG00000234616", "ENSG00000234741", "ENSG00000235194", "ENSG00000235711", "ENSG00000236279", "ENSG00000236287", "ENSG00000237190", "ENSG00000237515", "ENSG00000237765", "ENSG00000239305", "ENSG00000239306", "ENSG00000240230", "ENSG00000240344", "ENSG00000240583", "ENSG00000240682", "ENSG00000240694", "ENSG00000241258", "ENSG00000241553", "ENSG00000241878", "ENSG00000242247", "ENSG00000242259", "ENSG00000242265", "ENSG00000242485", "ENSG00000242802", "ENSG00000242808", "ENSG00000243147", "ENSG00000243156", "ENSG00000243943", "ENSG00000244005", "ENSG00000244187", "ENSG00000244405", "ENSG00000244462", "ENSG00000244754", "ENSG00000245532", "ENSG00000245910", "ENSG00000247077", "ENSG00000247556", "ENSG00000247596", "ENSG00000248092", "ENSG00000249242", "ENSG00000250317", "ENSG00000251022", "ENSG00000251562", "ENSG00000253352", "ENSG00000253719", "ENSG00000253729", "ENSG00000253738", "ENSG00000254004", "ENSG00000254377", "ENSG00000254858", "ENSG00000254999", "ENSG00000255248", "ENSG00000255302", "ENSG00000255529", "ENSG00000257218", "ENSG00000257727", "ENSG00000257923", "ENSG00000259974", "ENSG00000260230", "ENSG00000260916", "ENSG00000261115", "ENSG00000261824", "ENSG00000263753", "ENSG00000264364", "ENSG00000272047", "ENSG00000273079"], "row_sum": [-4988.0652, -2407.9361, 4132.0134, -4930.8169, 2853.9923, -1842.0487, 2842.7281, -2172.9927, 945.5594, 568.7723, -3587.2577, 2020.1643, -4190.8847, 5225.699, 2768.8062, 734.9595, -181.2036, 1430.98, -174.5167, -5219.3183, 1771.1531, 4400.2129, 2741.7576, 2545.2367, -4154.4639, 2428.1445, -1791.4098, 1716.0873, -634.1803, 2469.7918, -2324.5755, -4060.0567, 1150.461, 1260.2532, -1993.2218, -2650.3746, -1869.6146, 1879.7904, 1332.141, 1954.234, -1533.025, 1741.9472, 1596.1349, -3482.6393, 1187.0488, -701.7761, 850.6709, 1292.3233, 1165.6283, -1444.8478, -1950.8367, 767.4992, -2322.1471, 1699.0454, 655.1907, -614.4633, 1094.2468, 980.8849, -226.5458, 53.2622, -1447.5061, 1380.7509, -740.8461]}''')

In [ ]:
# ============================== LOAD AND CHECK ==============================
if ON_KAGGLE:
    z = np.load(find_input("18_hvg_expression.npz"), allow_pickle=True)
    Xk = pd.DataFrame(z["X"].astype(float), index=z["person"].astype(str), columns=[str(g) for g in z["genes"]])
    X = Xk.loc[REF["person"], REF["genes"]].to_numpy()
    yk = pd.Series(z["y"].astype(int), index=z["person"].astype(str)).loc[REF["person"]].to_numpy()
else:
    z = np.load(find_input("mdata.npz"), allow_pickle=True)
    X = z["X"].astype(float); yk = z["y"].astype(int)
y = np.array(REF["y"]); DS = np.array(REF["ds"]); PERSON = np.array(REF["person"]); GENES = list(REF["genes"])
diff = np.abs(X.sum(1) - np.array(REF["row_sum"])).max()
assert diff < 0.05 and (yk == y).all(), f"matrix differs from the reference ({diff:.4f})"
print("matrix identical to the reference copy (row sums and labels)")
STRATA = np.array([f"{d}_{v}" for d, v in zip(DS, y)])
XR = np.apply_along_axis(rankdata, 1, X) / X.shape[1]          # within-person ranks, label-free
SYM = pd.read_csv(find_input("15_gene_symbol_map.csv")).set_index("gene")["symbol"].to_dict()
sym = lambda g: SYM[g] if isinstance(SYM.get(g), str) else g
print(f"{len(y)} people ({(y == 0).sum()} control, {y.sum()} PD), {X.shape[1]:,} genes")
print(pd.crosstab(DS, np.where(y == 1, "PD", "control")).to_string())

In [ ]:
from scipy.stats import rankdata
from sklearn.ensemble import RandomForestClassifier
from sklearn.decomposition import PCA
from sklearn.metrics import (roc_auc_score, roc_curve, accuracy_score, balanced_accuracy_score, recall_score,
                             f1_score, matthews_corrcoef, brier_score_loss)

HEAD_NAME = "Ranks + PCA 30 + Random Forest"
HEAD = ("rpca", 30, dict(max_features=0.5, min_samples_leaf=1))

def rf(n_jobs=1, **kw):
    p = dict(n_estimators=N_TREES, max_features="sqrt", min_samples_leaf=1, class_weight="balanced",
             oob_score=True, n_jobs=n_jobs, random_state=SEED)
    p.update(kw); return RandomForestClassifier(**p)

def features(kind, arg, tr, te):
    if kind == "all":
        return X[tr], X[te]
    if kind == "genes":
        return X[tr][:, arg], X[te][:, arg]
    src = XR if kind == "rpca" else X
    p = PCA(arg, random_state=SEED).fit(src[tr])
    return p.transform(src[tr]), p.transform(src[te])

def oob_threshold(m, yt):
    """Accuracy cut-off chosen on the forest's out-of-bag predictions for the training people."""
    s = m.oob_decision_function_[:, 1]; ok = np.isfinite(s); s, yt = s[ok], yt[ok]
    u = np.unique(np.round(s, 6))
    cuts = np.concatenate([[-np.inf], (u[:-1] + u[1:]) / 2, [np.inf]]) if len(u) > 1 else np.array([0.5])
    acc = [accuracy_score(yt, (s > c).astype(int)) for c in cuts]
    best = np.flatnonzero(np.isclose(acc, max(acc)))
    return float(cuts[best[len(best) // 2]])

def fit_score(spec, tr, te, yy, n_jobs=1):
    kind, arg, kw = spec
    Xtr, Xte = features(kind, arg, tr, te)
    m = rf(n_jobs=n_jobs, **kw).fit(Xtr, yy[tr])
    return m.predict_proba(Xte)[:, 1], oob_threshold(m, yy[tr])

def metrics(yt, s, th=None):
    yh = (s >= 0.5).astype(int)
    out = {"auc": roc_auc_score(yt, s), "accuracy": accuracy_score(yt, yh), "bal_accuracy": balanced_accuracy_score(yt, yh),
           "sensitivity": recall_score(yt, yh), "specificity": recall_score(1 - yt, 1 - yh),
           "f1": f1_score(yt, yh, zero_division=0), "mcc": matthews_corrcoef(yt, yh), "brier": brier_score_loss(yt, s)}
    if th is not None:
        out["accuracy_oob_cut"] = accuracy_score(yt, (s > th).astype(int))
    return out

def summarise(name, recs, key):
    """Metrics of one model over the fold records: mean over CV repeats, pooled over LODO folds."""
    reps = sorted({o["rep"] for o in recs if o["kind"] == "cv"})
    per = []
    for r in reps:
        fr = [o for o in recs if o["kind"] == "cv" and o["rep"] == r]
        idx = np.concatenate([o["test"] for o in fr]); s = np.concatenate([o[key]["score"] for o in fr])
        th = np.concatenate([np.full(len(o["test"]), o[key]["thr"]) for o in fr])
        per.append(metrics(y[idx], s, th))
    d = pd.DataFrame(per)
    lo = [o for o in recs if o["kind"] == "lodo"]
    li = np.concatenate([o["test"] for o in lo]); ls = np.concatenate([o[key]["score"] for o in lo])
    row = {"model": name, **{f"cv_{k}": d[k].mean() for k in d.columns},
           "cv_auc_sd": d.auc.std(ddof=1) if len(d) > 1 else np.nan,
           "cv_accuracy_sd": d.accuracy.std(ddof=1) if len(d) > 1 else np.nan,
           "lodo_auc": roc_auc_score(y[li], ls), "lodo_accuracy": accuracy_score(y[li], (ls >= 0.5).astype(int))}
    for o in lo:
        row[f"lodo_auc_{o['tag'][5:]}"] = roc_auc_score(y[np.array(o["test"])], o[key]["score"])
    return row

def tree_shap(model, Z):
    v = shap.TreeExplainer(model).shap_values(Z)
    v = v[1] if isinstance(v, list) else v
    return v[..., 1] if v.ndim == 3 else v

def head_explain(idx):
    """Fit the headline model on idx. SHAP on its components, shared out to genes by squared loadings
    (each loading vector has unit length, so a component's importance is split, not created)."""
    p = PCA(30, random_state=SEED).fit(XR[idx]); Z = p.transform(XR[idx])
    m = rf(n_jobs=N_CPU, **HEAD[2]).fit(Z, y[idx])
    phi = tree_shap(m, Z)
    axis_imp = np.abs(phi).mean(0)
    gene_imp = (p.components_ ** 2 * axis_imp[:, None]).sum(0)
    push = phi.sum(1)                                                # each person's SHAP push towards PD
    Rg = np.apply_along_axis(rankdata, 0, XR[idx]); Rt = rankdata(push)
    Rg = (Rg - Rg.mean(0)) / (Rg.std(0) + 1e-12); Rt = (Rt - Rt.mean()) / Rt.std()
    rho = (Rg * Rt[:, None]).mean(0)                                 # Spearman: gene rank vs PD push
    return p, m, phi, axis_imp, gene_imp, rho

## 1. The folds every notebook uses

5 x 5-fold cross-validation by person, stratified by study and diagnosis (seeds 1000-1004,
never used in the sweep), plus leave one study out. Saved so every secondary notebook
scores on exactly the same people.

In [ ]:
FOLDS = []
for r in range(N_REP):
    for k, (tr, te) in enumerate(StratifiedKFold(5, shuffle=True, random_state=1000 + r).split(X, STRATA)):
        FOLDS.append({"tag": f"r{r}k{k}", "kind": "cv", "rep": r, "seed": 1000 + 100 * r + k, "train": tr, "test": te})
for d in sorted(set(DS)):
    FOLDS.append({"tag": f"lodo_{d}", "kind": "lodo", "rep": -1, "seed": SEED,
                  "train": np.where(DS != d)[0], "test": np.where(DS == d)[0]})
json.dump([{**f, "train": f["train"].tolist(), "test": f["test"].tolist()} for f in FOLDS],
          open(OUT / "core_folds.json", "w"))
print(len(FOLDS), "folds saved")

## 2. How well it classifies

The headline model and, for reference, a plain Random Forest on all 5,622 genes, on the
same folds. Accuracy uses a 0.5 cut; `accuracy_oob_cut` uses a cut chosen on the
training people's out-of-bag predictions. For every CV fold the model is also explained
on its training people, to measure how stable its top genes are.

In [ ]:
def run_fold(f):
    tr, te = f["train"], f["test"]
    rec = {k: (v.tolist() if isinstance(v, np.ndarray) else v) for k, v in f.items() if k != "train"}
    s, t = fit_score(HEAD, tr, te, y); rec["head"] = {"score": s.tolist(), "thr": t}
    s, t = fit_score(("all", None, {}), tr, te, y); rec["rf_all"] = {"score": s.tolist(), "thr": t}
    return rec
REC = Parallel(n_jobs=N_CPU)(delayed(run_fold)(f) for f in FOLDS)
for rec, f in zip(REC, FOLDS):
    rec["test"] = np.array(rec["test"])
    if f["kind"] == "cv":
        *_, gene_imp, _ = head_explain(f["train"])
        rec["head_top100"] = [GENES[j] for j in np.argsort(-gene_imp)[:100]]
PERF = pd.DataFrame([summarise(HEAD_NAME, REC, "head"), summarise("Random Forest, all genes", REC, "rf_all")])
PERF.to_csv(OUT / "core_performance.csv", index=False)
json.dump([{**r, "test": r["test"].tolist()} for r in REC], open(OUT / "core_fold_records.json", "w"))
pd.set_option("display.width", 220)
print(PERF[["model", "cv_auc", "cv_auc_sd", "cv_accuracy", "cv_accuracy_sd", "cv_accuracy_oob_cut", "cv_bal_accuracy",
            "cv_sensitivity", "cv_specificity", "lodo_auc"]].round(3).to_string(index=False))

oof = np.zeros(len(y)); cnt = np.zeros(len(y))
for rec in REC:
    if rec["kind"] == "cv":
        oof[rec["test"]] += rec["head"]["score"]; cnt[rec["test"]] += 1
oof /= np.maximum(cnt, 1)
pd.DataFrame({"person": PERSON, "dataset": DS, "y": y, "oof_score": oof}).to_csv(OUT / "core_oof_scores.csv", index=False)
fpr, tpr, _ = roc_curve(y, oof)
pd.DataFrame({"fpr": fpr, "tpr": tpr}).to_csv(OUT / "core_roc_curve.csv", index=False)
log(f"out-of-fold AUC of the averaged scores: {roc_auc_score(y, oof):.3f}")

## 3. The final model on all 63 people, and what drives it

In [ ]:
pH, mH, phiH, axis_imp, gene_imp, rho = head_explain(np.arange(len(y)))
joblib.dump({"pca": pH, "rf": mH, "genes": GENES, "note": "input = within-person ranks of the z-scored genes / n_genes"},
            OUT / "core_model.joblib")
head_rank = pd.Series(gene_imp, index=GENES).rank(ascending=False, method="first").astype(int)
cv_recs = [r for r in REC if r["kind"] == "cv"]
freq50 = pd.Series(0.0, index=GENES)
for r in cv_recs:
    freq50[r["head_top100"][:50]] += 1
freq50 /= len(cv_recs)
G = pd.DataFrame({"gene": GENES, "symbol": [sym(g) for g in GENES], "importance": gene_imp,
                  "rank": head_rank.to_numpy(), "direction_rho": rho,
                  "direction": np.where(rho > 0, "higher rank -> more PD-like", "higher rank -> less PD-like"),
                  "top50_fold_frequency": freq50.to_numpy()}).sort_values("rank")
G.to_csv(OUT / "core_gene_importance.csv", index=False)
C = pd.DataFrame({"component": [f"PC{k + 1}" for k in range(len(axis_imp))], "mean_abs_shap": axis_imp,
                  "variance_explained": pH.explained_variance_ratio_}).sort_values("mean_abs_shap", ascending=False)
C.to_csv(OUT / "core_component_importance.csv", index=False)
pd.DataFrame(phiH, columns=[f"PC{k + 1}" for k in range(phiH.shape[1])]).assign(person=PERSON, dataset=DS, y=y) \
  .to_csv(OUT / "core_component_shap.csv", index=False)
print(C.head(8).round(4).to_string(index=False))
print(G.head(25)[["rank", "symbol", "importance", "direction_rho", "top50_fold_frequency"]].round(4).to_string(index=False))

In [ ]:
# ============================== SAVE FOR THE SECONDARY NOTEBOOKS ==============================
np.savez_compressed(OUT / "core_data.npz", X=X, XR=XR, y=y, ds=DS, person=PERSON, genes=np.array(GENES))
for f in ("03_de_results_full.csv", "15_gene_symbol_map.csv", "01_preprocessing_summary.csv", "01_cohort_by_dataset.csv"):
    pd.read_csv(find_input(f)).to_csv(OUT / f, index=False)
hp = PERF.set_index("model").loc[HEAD_NAME]
json.dump({"model": HEAD_NAME, "spec": {"features": "within-person ranks -> PCA(30), fitted on training people",
                                        "forest": {"n_estimators": N_TREES, "max_features": 0.5, "min_samples_leaf": 1,
                                                   "class_weight": "balanced"}},
           "n_people": int(len(y)), "n_genes": int(X.shape[1]),
           "cv_auc": float(hp.cv_auc), "cv_auc_sd": float(hp.cv_auc_sd), "cv_accuracy": float(hp.cv_accuracy),
           "cv_accuracy_sd": float(hp.cv_accuracy_sd), "lodo_auc": float(hp.lodo_auc),
           "chosen_in": "alisaremi/pd-lcm-rf-confirm (97 Random Forest variants)"},
          open(OUT / "core_summary.json", "w"), indent=1)
print(sorted(p.name for p in OUT.iterdir()))
log("core done")